## Variant region list:
- load the file of all sequences/oligo names (80215)
- make a list of these with a column for the name, sequence, type (if variant: ALT, if reference: REF, if region: region, control (true/false), gene association (all information about the gene (i.e. name with | and ~RAS ...))
- example table is shown [here](https://docs.google.com/spreadsheets/d/1YfepW_nv14024v8KwveGaJTbcBX6pRKRUohl7jZhltY/edit#gid=0)
### Questions:
- Some rows have ALT_, REF_, what does it mean if it has no REF_ and ALT_ is it a region then?
  - (1) id (not too long), (2) sequence, (3) category,  (4) class (test, variant (pos/neg) control), (5) source ? for our design ("candidate CRE nearby 536 cardiac, neuro, cava and random genes"), (6) ref_sequence ? CLEA controlls? what with the sequences (general controlls we took them actually from hg19, I think?,  (7) chrom (NA possible), (8) chrom_start (NA possible), (9) chrom_end (NA possible),  (10) variant_class (NA possible), (11) variant_pos (NA possible), (12) SPDI (NA possible), (13) allele (NA possible),  (14) info (free form)

### Process:
- We identified regions near TSS of genes (we looked for variants within these regions (centered in these regions (100bp from the center)))
- Total number of different regions: 80215
- First table:
  - for each row: one header with the informations it has ()


In [1]:
# imports 
import pandas as pd 
from Bio import SeqIO
# use the config file in yaml format
import yaml
import os
import re
import gzip # gzipped files
import hashlib

# read config
config_path = "/data/gpfs-1/users/kisa11_c/work/coding/80K_analysis/05_variant_region_list/config/config.yaml"
config_path = "./config/config.yaml"
with open(config_path, 'r') as ymlfile:
    config = yaml.safe_load(ymlfile)

In [2]:
# load the data
design_fasta = '/fast/groups/ag_kircher/MPRA/IGVF_Y1_design/resources/association_data/design_no_duplicates_sequence_and_header.fa'
design_fasta = 'resources/design_no_duplicates_sequence_and_header.fa'
design_fasta = config['files']['reference']

# read the fasta file with the sequences and prepare a tsv with header and sequence using biopython

records = list(SeqIO.parse(design_fasta, "fasta"))
design_df = pd.DataFrame(columns=['header', 'sequence'])
header = [] 
sequence = []
for record in records:
    header.append(record.id)
    sequence.append(str(record.seq))

design_df['header'] = header
design_df['sequence'] = sequence

# label is the string in front of the first ":"
design_df['label'] = design_df['header'].str.split(':').str[0]
cardiac_neuro_cava_random = design_df[design_df['label'] == 'cardiac_neuro_cava_random']

design_df

,header,sequence,label
0,cardiac_neuro_cava_random:SKI|ENSG00000157933....,AGGACCGGATCAACTAAGAATACAAGTAACTGATGAATGAAGGGGG...,cardiac_neuro_cava_random
1,cardiac_neuro_cava_random:SKI|ENSG00000157933....,AGGACCGGATCAACTTTGGGTATGCTGCCCCCCAGCTGGCGGGGCA...,cardiac_neuro_cava_random
2,cardiac_neuro_cava_random:SKI|ENSG00000157933....,AGGACCGGATCAACTACGAGCAAGGGAATGAGAGAGAGTGGGTTAG...,cardiac_neuro_cava_random
3,cardiac_neuro_cava_random:SKI|ENSG00000157933....,AGGACCGGATCAACTCGTGGACACGCGTGATTGACCCTTTAACTGT...,cardiac_neuro_cava_random
4,cardiac_neuro_cava_random:SKI|ENSG00000157933....,AGGACCGGATCAACTCCGGAGAGTCTCAGCTCCCGCAGCCCTAACA...,cardiac_neuro_cava_random
...,...,...,...
80210,MK:tile_2240|chr1-116244322+116244591|scramble...,AGGACCGGATCAACTCTTAATCAAATAACCCATTAATTCTATATAT...,MK
80211,MK:tile_6675|chr11-2374617+2374886|scramble_ne...,AGGACCGGATCAACTCATCGGCCCTGGTGAAGCGTCCGTCCAGACG...,MK
80212,MK:tile_18415|chr17-71181691+71181960|scramble...,AGGACCGGATCAACTTAAATATTCAGCGATACATTCCTATTCTTTT...,MK
80213,MK:tile_14356|chr15-67031618+67031887|scramble...,AGGACCGGATCAACTTGAAGCCCCTGATTCTGTTAGAATAAGGTTA...,MK


### Trying to find unmerged and merged headers (does not work properly)

In [36]:
# read unmerged file
unmerged_file = "/home/kisa/coding/80K_MPRA/80K-Analysis/05_variant_region_list/resources/design.fa"
unmerged_file = config['files']['duplicate_reference']

# if file is gzipped, use the following
if unmerged_file.endswith(".gz"):
    file_handler = gzip.open(unmerged_file, "rt")
else:
    file_handler = open(unmerged_file, "r")

records = list(SeqIO.parse(file_handler, "fasta"))
unmer_design_df = pd.DataFrame(columns=['header', 'sequence'])
header = [] 
sequence = []
for record in records:
    header.append(record.id)
    sequence.append(str(record.seq))

unmer_design_df['header'] = header
unmer_design_df['sequence'] = sequence
unmer_design_df["unmerged"] = True
unmer_design_df = unmer_design_df[['header', 'unmerged']]
unmer_design_df.head()

,header,unmerged
0,cardiac_neuro_cava_random:SKI|ENSG00000157933....,True
1,cardiac_neuro_cava_random:SKI|ENSG00000157933....,True
2,cardiac_neuro_cava_random:SKI|ENSG00000157933....,True
3,cardiac_neuro_cava_random:SKI|ENSG00000157933....,True
4,cardiac_neuro_cava_random:SKI|ENSG00000157933....,True


In [37]:
# merge unmer_design_df and design_df on header
# the lines without label are merged
complete_design_df = design_df.merge(unmer_design_df, on="header", how="left")
complete_design_df
# number of na values in "label"
merged_header_df = complete_design_df[complete_design_df.unmerged.isna()] #3278
merged_header_df.shape
merged_header_list = merged_header_df.header.to_list()
merged_header_list[100]
# example 'cardiac_neuro_cava_random:REF_CSDE1|ENSG00000009307.17|EH38E1378368~NRAS|ENSG00000213281.5|EH38E1378368_rev_tile1-1'

'cardiac_neuro_cava_random:REF_CSDE1|ENSG00000009307.17|EH38E1378368~NRAS|ENSG00000213281.5|EH38E1378368_rev_tile1-1'

In [38]:
merged_header_df

,header,sequence,label,unmerged
708,cardiac_neuro_cava_random:CSDE1|ENSG0000000930...,AGGACCGGATCAACTAAGGAAGGGAGGGAGGGAGGGAGCGATCCCT...,cardiac_neuro_cava_random,NaN
709,cardiac_neuro_cava_random:CSDE1|ENSG0000000930...,AGGACCGGATCAACTGGACCACCTCCACCAACTGTCAGCTCACATC...,cardiac_neuro_cava_random,NaN
710,cardiac_neuro_cava_random:CSDE1|ENSG0000000930...,AGGACCGGATCAACTACTGTCCTTCCTGAGGCCTCCAGCATTATTG...,cardiac_neuro_cava_random,NaN
711,cardiac_neuro_cava_random:CSDE1|ENSG0000000930...,AGGACCGGATCAACTTTACTTATTTATTTATTTTTTGAGACAGGGT...,cardiac_neuro_cava_random,NaN
712,cardiac_neuro_cava_random:CSDE1|ENSG0000000930...,AGGACCGGATCAACTACCCCATAGTATGCCTCCCTCCCTCTCTCCT...,cardiac_neuro_cava_random,NaN
...,...,...,...,...
77523,C_positive_heart_AB:FLNA-ENST00000420627.5:Oli...,AGGACCGGATCAACTGGTCGCCCATTCCCAAGCTCCCACCTTGACG...,C_positive_heart_AB,NaN
77524,C_positive_heart_AB:FLNA-ENST00000420627.5:Oli...,AGGACCGGATCAACTGGGCCCAACCAAGGAACCTGGCCTGGTCTCA...,C_positive_heart_AB,NaN
77525,C_positive_heart_AB:FLNA-ENST00000420627.5:Oli...,AGGACCGGATCAACTCCACTCGCCCTGAGTCCACACAAGTTCCTGG...,C_positive_heart_AB,NaN
77526,C_positive_heart_AB:FLNA-ENST00000360319.9:Oli...,AGGACCGGATCAACTGTAAAATTGCCCAGGAGCCCGGGACGGGTGC...,C_positive_heart_AB,NaN


In [39]:
# count number of rows per label
# split based on "|" number in order to find pattern
merged_header_df["pipe_count"] = merged_header_df['header'].str.count(r'\|')
merged_header_df_cardiac = merged_header_df[merged_header_df["label"] == "cardiac_neuro_cava_random"]
merged_header_df_cardiac #1717 / 3278 => 52%
merged_header_df_cardiac.pipe_count.value_counts()
# pipe_count
# 8     596
# 7     471
# 4     341
# 10    309

/data/gpfs-1/users/kisa11_c/scratch/tmp/hpc-cpu-82/ipykernel_1569958/2263432361.py:3: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  merged_header_df["pipe_count"] = merged_header_df['header'].str.count(r'\|')


pipe_count
8     596
7     471
4     341
10    309
Name: count, dtype: int64

### Split the number of headers from cardiac_neuro_cava_random label by the number of pipes to find patterns
- found patterns
  - alternate oligos: ALT_ (one snp each)
  - reference oligos: REF_ (some are merged) also regions
  - regions: no REF_ or ALT_ (some are merged)

#### Split of cardiac_neuro_cava_random by number of pipes:
- pipe_count
- 8     596
- 7     471
- 4     341
- 10    309

##### 8 pipes: (596)
- all alt
- do all have same number of "_" 
- do all have same number and position of "~"
- do all end with a variant
- example: # 'cardiac_neuro_cava_random:ALT_DRD4|ENSG00000069696.7|EH38E2937745_fwd_tile1-1_DEAF1|ENSG00000177030.19|EH38E2937745|11-596480-T-C~DRD4|ENSG00000069696.7|EH38E2937745|11-596480-T-C'

In [31]:
# 8: 596 rows
cardiac_8_pipe_df = merged_header_df_cardiac[merged_header_df_cardiac["pipe_count"] == 8]
cardiac_8_pipe_df # 596

# count number of "_"
cardiac_8_pipe_df.header.str.count(r"_")
# count number and look at position of "~"

# end pattern matches variant patter?
cardiac_8_pipe_list = cardiac_8_pipe_df["header"].to_list()
cardiac_8_pipe_list[0]


'cardiac_neuro_cava_random:ALT_DRD4|ENSG00000069696.7|EH38E2937745_fwd_tile1-1_DEAF1|ENSG00000177030.19|EH38E2937745|11-596480-T-C~DRD4|ENSG00000069696.7|EH38E2937745|11-596480-T-C'

#### Short cut ideas:
- filter all header with variants: |(last pipe)<num>-<num>
  - Pattern ? if variant is there there is ALT in the beginning
- Find variant pattern -> its variant
- Find reference pattern -> its reference
- Find nothing -> its region
- check the numbers

In [4]:

def check_variant(header):
    """Checks if a header is a variant header"""
    # get the last part of the header
    last_part = header.split('|')[-1]
    # check if the last part if it matches the regex [\d]+-[\d]+
    if re.match(r'[\dA-Z]+-[\d]+', last_part): # is sufficient, because all these headers have ALT in their name
        return True
    else:
        return False

def check_reference(header):
    """Checks if a header is a reference header"""
    # check after the label if REF_ is in the header
    non_label_header = header.split(':')[1]
    if "REF_" in non_label_header.split('|')[0]:
        return True
    else:
        return False

def check_region(header):
    """
    Checks if a header is a region header
    A region header is here defined as a header without ALT_ or REF_ after the first ":" 
    """
    # check after the label if REF_ is in the header
    non_label_header = header.split(':')[1]
    if "REF_" in non_label_header.split('|')[0]:
        return False
    elif "ALT_" in non_label_header.split('|')[0]:
        return False
    else:
        return True

def check_region_variant_reference_numbers(header_list):
    """Checks if a header is a variant, reference or region header"""
    ref_counter = 0
    var_counter = 0
    region_counter = 0
    unknown_counter = 0
    for header in header_list:
        if header.split(':')[0] != 'cardiac_neuro_cava_random':
            continue
        if check_variant(header):
            var_counter += 1
            header_type = 'ALT'
        elif check_reference(header):
                ref_counter += 1
                header_type = 'REF'
        elif check_region(header):    
            region_counter += 1
            header_type = 'region'
        else:
            header_tpye = 'unknown'
            unknown_counter += 1
            print(f'Found unknown header: {header}')
    return ref_counter, var_counter, region_counter

def create_fasta_df_from_one_line_sequence_fasta(fasta_path, filter_cardiac=False):
    """Creates a dataframe with header and sequence from a fasta file which has one line per sequence"""
    records = list(SeqIO.parse(fasta_path, "fasta"))
    design_df = pd.DataFrame(columns=['header', 'sequence'])
    header = [] 
    sequence = []
    for record in records:
        header.append(record.id)
        sequence.append(str(record.seq))

    design_df['header'] = header
    design_df['sequence'] = sequence

    if filter_cardiac:
        design_df['label'] = design_df['header'].str.split(':').str[0]
        design_df = design_df[design_df['label'] == 'cardiac_neuro_cava_random']

    return design_df

def get_gene_name_from_header(header):
    """Identify the gene name (e.g. MYH6) from the header: sequence after first ":" and before first "|" then crop the sequence after the first "_" if it exists"""
    gene_name = header.split(':')[1].split('|')[0]
    if '_' in gene_name:
        gene_name = gene_name.split('_')[1]
    return gene_name

def write_variants_fasta(design_fasta_path, fasta_out_directory):
    """Write all variants to a fasta file with the header and sequence"""
    design_df = create_fasta_df_from_one_line_sequence_fasta(design_fasta_path, filter_cardiac=False)
    design_df['is_variant'] = design_df['header'].apply(check_variant)
    design_df = design_df[design_df['is_variant'] == True]
    # write the fasta file
    output_path = os.path.join(fasta_out_directory, f'identified_variants_{design_df.shape[0]}.fa')
    with open(output_path, 'w') as f:
        for index, row in design_df.iterrows():
            f.write('>' + row['header'] + '\n' + row['sequence'] + '\n')
    return design_df

def write_region_fasta(design_fasta_path, fasta_out_directory):
    """Write all regions to a fasta file with the header and sequence"""
    design_df = create_fasta_df_from_one_line_sequence_fasta(design_fasta_path, filter_cardiac=True)
    design_df['is_region'] = design_df['header'].apply(check_region)
    design_df = design_df[design_df['is_region'] == True]
    # write the fasta file
    output_path = os.path.join(fasta_out_directory, f'identified_regions_{design_df.shape[0]}.fa')
    with open(output_path, 'w') as f:
        for index, row in design_df.iterrows():
            f.write('>' + row['header'] + '\n' + row['sequence'] + '\n')
    return design_df

def write_reference_fasta(design_fasta_path, fasta_out_directory):
    """Write all references to a fasta file with the header and sequence"""
    design_df = create_fasta_df_from_one_line_sequence_fasta(design_fasta_path, filter_cardiac=True)
    design_df['is_reference'] = design_df['header'].apply(check_reference)
    design_df = design_df[design_df['is_reference'] == True]
    # write the fasta file
    output_path = os.path.join(fasta_out_directory, f'identified_references_{design_df.shape[0]}.fa')
    with open(output_path, 'w') as f:
        for index, row in design_df.iterrows():
            f.write('>' + row['header'] + '\n' + row['sequence'] + '\n')
    return design_df



In [33]:
# iterate all headers and get the gene name of interest
gene_names = set()
not_cardiac = 0
for hdr in header:
    if hdr.split(':')[0] != 'cardiac_neuro_cava_random':
        not_cardiac += 1
        continue
    # put gene name into set
    gene_name = get_gene_name_from_header(hdr)
    gene_names.add(gene_name)

# check the number of the gene_name set
print(f'Number of unique "assiciated" genes: {len(gene_names)}')  # => Found all (same number as in summary presentation) 525 "associated" genes

Number of unique "assiciated" genes: 525


In [46]:
print(not_cardiac)
num_cardiac = 80215 - 6275
print(num_cardiac)

asdf = 18582 + 46458 + 8900
asdf

6275
73940


73940

In [87]:

ref_counter, var_counter, region_counter = check_region_variant_reference_numbers(header)
print(ref_counter, var_counter, region_counter) # 18582 + 46458 + 8900 = 73940
# we want: 28000 cCREs we got 8900
design_fasta_file = config['files']['reference']
output_fasta_directory = 'resources/'
write_variants_fasta(design_fasta_file, output_fasta_directory)
write_reference_fasta(design_fasta_file, output_fasta_directory)
write_region_fasta(design_fasta_file, output_fasta_directory)
# Problem: we are not sure about the exact numbers and the formats might be different
  # - how do I check if the number of variants (currently 46458) is correct?

18582 46458 8900


,header,sequence,label,is_region
0,cardiac_neuro_cava_random:SKI|ENSG00000157933....,AGGACCGGATCAACTAAGAATACAAGTAACTGATGAATGAAGGGGG...,cardiac_neuro_cava_random,True
1,cardiac_neuro_cava_random:SKI|ENSG00000157933....,AGGACCGGATCAACTTTGGGTATGCTGCCCCCCAGCTGGCGGGGCA...,cardiac_neuro_cava_random,True
2,cardiac_neuro_cava_random:SKI|ENSG00000157933....,AGGACCGGATCAACTACGAGCAAGGGAATGAGAGAGAGTGGGTTAG...,cardiac_neuro_cava_random,True
3,cardiac_neuro_cava_random:SKI|ENSG00000157933....,AGGACCGGATCAACTCGTGGACACGCGTGATTGACCCTTTAACTGT...,cardiac_neuro_cava_random,True
4,cardiac_neuro_cava_random:SKI|ENSG00000157933....,AGGACCGGATCAACTCCGGAGAGTCTCAGCTCCCGCAGCCCTAACA...,cardiac_neuro_cava_random,True
...,...,...,...,...
8895,cardiac_neuro_cava_random:FLNA|ENSG00000196924...,AGGACCGGATCAACTTTGGACTCTGGGCTGCTCAGAGGCTGCCTTG...,cardiac_neuro_cava_random,True
8896,cardiac_neuro_cava_random:FLNA|ENSG00000196924...,AGGACCGGATCAACTAGAGCCCTGGGGAACGCCATGAGCCCTCAGG...,cardiac_neuro_cava_random,True
8897,cardiac_neuro_cava_random:FLNA|ENSG00000196924...,AGGACCGGATCAACTTTGGGACTTAAACCCCAGCCTCCCCCGTCCA...,cardiac_neuro_cava_random,True
8898,cardiac_neuro_cava_random:FLNA|ENSG00000196924...,AGGACCGGATCAACTAGCCCATAATTTATTGATTTTTTAAAATTTG...,cardiac_neuro_cava_random,True


In [41]:
# read all headers and check for the regex pattern of variant info after the last pipe ("|")
# count the number of found variant patterns
# if the pattern is found, check if it has ALT_ after the first ":"
# count the number of found variant pattersn with ALT_
var_count = 0
var_count_alt = 0
for hdr in header:
    if hdr.split(':')[0] != 'cardiac_neuro_cava_random':
        continue
    if re.match('ALT_', hdr.split(':')[1]):
        # if it matches, count it
        var_count_alt += 1
        # print(hdr)
        # break
        # get the last part of the header
        last_part = hdr.split('|')[-1]
        # check if the last part if it matches the regex [\d]+-[\d]+
        if re.match(r'[\dA-Z]+-[\d]+', last_part):
            var_count += 1
        else:
            print(hdr)
        
# check if the number of variants is the same as the number of variants with ALT_
if var_count == var_count_alt:
    print("All variants have ALT_ in the header")

All variants have ALT_ in the header


In [6]:
# list of all unique labels
unique_labels = design_df['label'].unique().tolist()
unique_labels

['cardiac_neuro_cava_random',
 'GC_Atrial_fib',
 'GC_Liang',
 'GC_Selvarajan',
 'GC_Mohlke',
 'GC_Kircher',
 'GC_Mendelian_variants',
 'C_positive_heart_CAD',
 'GC_Cort_Chengyu',
 'GC_GABA_Chengyu',
 'GC_Glut_Chengyu',
 'GC_Hon',
 'GC_Vista',
 'GC_DNase_positive',
 'GC_DNase_negative_brain',
 'GC_DNase_negative_blood',
 'C_negative_heart_MK',
 'C_negative_neuron_MK',
 'C_negative_neuron_NP',
 'C_positive_heart_MK',
 'C_positive_neuron_CD',
 'C_positive_neuron_MK',
 'C_positive_neuron_NP',
 'C_positive_heart_AB',
 'C_SLEA',
 'GC_DNase_positive_shuffeled',
 'GC_DNase_negative_brain_shuffeled',
 'GC_DNase_negative_blood_shuffeled',
 'MK']

In [7]:
# investigate the variants
for i in range(40, 48):
    print(design_df['header'].tolist()[i])


cardiac_neuro_cava_random:PRDM16|ENSG00000142611.17|EH38E2779571_fwd_tile1-1
cardiac_neuro_cava_random:PRDM16|ENSG00000142611.17|EH38E2779574_fwd_tile1-1
cardiac_neuro_cava_random:PRDM16|ENSG00000142611.17|EH38E2779580_fwd_tile1-1
cardiac_neuro_cava_random:PRDM16|ENSG00000142611.17|EH38E2779583_fwd_tile1-1
cardiac_neuro_cava_random:PRDM16|ENSG00000142611.17|EH38E2779643_fwd_tile1-1
cardiac_neuro_cava_random:PRDM16|ENSG00000142611.17|EH38E2779653_fwd_tile1-1
cardiac_neuro_cava_random:PRDM16|ENSG00000142611.17|EH38E2779655_fwd_tile1-1
cardiac_neuro_cava_random:PRDM16|ENSG00000142611.17|EH38E2779714_fwd_tile1-1


In [40]:
for lable in unique_labels:
    if lable == 'cardiac_neuro_cava_random':
        continue
    df = design_df[design_df['label'] == lable]
    print(lable)
    print(df['header'].str.count('\|').value_counts())

NameError: name 'unique_labels' is not defined

### Metadata file: [document](https://docs.google.com/document/d/1ThHgLjMnS2r-vv_4ZHHO9y4K1Qbc2S2WKKUDw__iYXU/edit)
- I want to have a table of id, sequence, category, class, source, ref_sequence, chrom, chrom_start, chrom_end, variant_class, variant_pos, SPDI, allele, info

In [6]:
import os
import pandas as pd
import numpy as np
focusing_label = 'cardiac_neuro_cava_random'

def check_variant(header, focusing_label=""):
    """Checks if a header is a variant header"""
    if focusing_label == 'cardiac_neuro_cava_random': # asume variant has ALT_ or REF_ in header
        if "ALT_" in header or "REF_" in header:
            return True
    # TODO: add another pattern from the headers describing a variant
    return False    
   
def check_element(header, focusing_label=""):
    """Checks if a header is an element header"""
    if focusing_label == 'cardiac_neuro_cava_random': # asume element has no ALT_ or REF_ in header
        if "ALT_" not in header and "REF_" not in header:
            return True
    # TODO: add another pattern from the headers describing an element
    return False

def get_category(header):
    """Get the category of the header"""
    if focusing_label != "":      
        if focusing_label not in header:
            return 'NA'
    if check_variant(header, focusing_label):
        return 'variant'
    if check_element(header, focusing_label):
        return 'element'
    return 'NA'

def get_class(header):
    """
    Get the class of the header
    A class is according to the IGVF metadata format: test, variant positive control, variant negative control, 
          element active control or element inactive control
    """
    # known pattern for cardiac_neuro_cava_random: all are "test"
    if 'cardiac_neuro_cava_random' in header:
        return 'test'
    else: # TODO: add other patterns for the other 4 cases
        return 'NA'
    
def get_source(header):
    """Get the source of the header"""
    # only known pattern: for cardiac_neuro_cava_random: "candidate CRE nearby 536 cardiac, neuro, cava and random genes"
    if 'cardiac_neuro_cava_random' in header:
        return 'candidate CRE nearby cardiac, neuro, cava and random genes'
    if 'GC_' in header:
        return 'IGVF general controls'
    else:
        return 'NA'
    
def create_path(path):
    """Check if path exists if not create it"""
    dir_path = os.path.dirname(path)
    if not os.path.exists(dir_path):
        os.makedirs(dir_path)
        
    

In [7]:
# load fasta (header, sequence) and add columns with NA values 
design_fasta = config['files']['reference']
pre_metadata_df = create_fasta_df_from_one_line_sequence_fasta(design_fasta, filter_cardiac=False)

# add empty columns id, category, class, source, ref_sequence, chrom, chrom_start, chrom_end, variant_class, variant_pos, SPDI, allele, info

pre_metadata_df['id'] = 'NA' # unique identifier: oligo_<id> while id is the md5 hash of the sequence
pre_metadata_df['category'] = 'NA' # for cardiac_neuro_cava_random: if "ALT_" or "REF_" in header, then "variant", else "element" otherwise leave NA
pre_metadata_df['class'] = 'NA' # if cardiac_neuro_cava_random: "test", else leave NA (TODO: find pattern in controls for variants and elements and positive and negative controls)
pre_metadata_df['source'] = 'NA' # if cardiac_neuro_cava_random: "candidate CRE nearby 536 cardiac, neuro, cava and random genes", if "GC" in header, then "IGVF general controls", else leave NA
pre_metadata_df['ref_sequence'] = 'NA' # if "cardiac_neuro_cava_random" then "hg38", if "CLEA" in header, then "hg18", else leave NA
pre_metadata_df['chrom'] = 'NA' # leave NA (TODO: add this from the regions.bed file in the final_design/*/final_design directory)
pre_metadata_df['chrom_start'] = 'NA' # leave NA (TODO: see above)
pre_metadata_df['chrom_end'] = 'NA' # leave NA
pre_metadata_df['variant_class'] = 'NA' # if "cardiac_neuro_cava_random" and category is "variant" then "SNV", else leave NA
pre_metadata_df['variant_pos'] = 'NA' # leave NA (TODO: see above + compute from reference position (pos of variant - 1) - pos of reference = variant_pos (0-based))
pre_metadata_df['SPDI'] = 'NA' # leave NA
pre_metadata_df['allele'] = 'NA' # if cardiac_neuro_cava_random if REF_ in header then "ref", if ALT_ in header then "alt", else leave NA
pre_metadata_df['info'] = 'NA' # leave NA

# id:
pre_metadata_df['id'] = pre_metadata_df['sequence'].apply(lambda x: 'oligo_' + hashlib.md5(x.encode()).hexdigest())
# check for duplicates
pre_metadata_df['id'].duplicated().sum() # 0

# category:
pre_metadata_df['category'] = pre_metadata_df['header'].apply(get_category)

# class:
pre_metadata_df['class'] = pre_metadata_df['header'].apply(get_class)

# source:
pre_metadata_df['source'] = pre_metadata_df['header'].apply(get_source)

# ref_sequence:
pre_metadata_df['ref_sequence'] = pre_metadata_df['header'].apply(lambda x: 'hg38' if 'cardiac_neuro_cava_random' in x else ('hg18' if 'CLEA' in x else 'NA'))

# leave NA for chrom, chrom_start, chrom_end, 

# variant_class:
pre_metadata_df['variant_class'] = pre_metadata_df.apply(lambda x: 'SNV' if x['category'] == 'variant' else 'NA', axis=1)

# leave NA for variant_pos, SPDI, 

# allele:
pre_metadata_df['allele'] = pre_metadata_df['header'].apply(lambda x: 'ref' if 'REF_' in x else ('alt' if 'ALT_' in x else 'NA'))

# leave info as NA

pre_metadata_df

# get the id and header for the metadata file
id_header_match = pre_metadata_df[['id', 'header']]

id_header_match_path = config['files']['id_header_match'] 


# write the match table to a tsv
# create output path
create_path(id_header_match_path)
id_header_match.to_csv(id_header_match_path, sep='\t', index=False)

# order the columns as in the metadata file
pre_metadata_df_final = pre_metadata_df[['id', 'sequence', 'category', 'class', 'source', 'ref_sequence', 'chrom', 'chrom_start', 'chrom_end', 'variant_class', 'variant_pos', 'SPDI', 'allele', 'info']]

pre_metadata_df_final

# # write the metadata file
# metadata_path = config['files']['metadata_table']
# create_path(metadata_path)
# pre_metadata_df.to_csv(metadata_path, sep='\t', index=False)

# in the end check NA distribution/number of each column 

,id,sequence,category,class,source,ref_sequence,chrom,chrom_start,chrom_end,variant_class,variant_pos,SPDI,allele,info
0,oligo_c32acb98ad2a851ab46621b1c3af8b44,AGGACCGGATCAACTAAGAATACAAGTAACTGATGAATGAAGGGGG...,element,test,"candidate CRE nearby cardiac, neuro, cava and ...",hg38,NA,NA,NA,NA,NA,NA,NA,NA
1,oligo_8deb96fab75f4f41dbc05b2d163ad89b,AGGACCGGATCAACTTTGGGTATGCTGCCCCCCAGCTGGCGGGGCA...,element,test,"candidate CRE nearby cardiac, neuro, cava and ...",hg38,NA,NA,NA,NA,NA,NA,NA,NA
2,oligo_d08942ae2ac12327ebaad04b395b7dc5,AGGACCGGATCAACTACGAGCAAGGGAATGAGAGAGAGTGGGTTAG...,element,test,"candidate CRE nearby cardiac, neuro, cava and ...",hg38,NA,NA,NA,NA,NA,NA,NA,NA
3,oligo_d0ac046887b1f97d9c494e6a2bae69f7,AGGACCGGATCAACTCGTGGACACGCGTGATTGACCCTTTAACTGT...,element,test,"candidate CRE nearby cardiac, neuro, cava and ...",hg38,NA,NA,NA,NA,NA,NA,NA,NA
4,oligo_328edd51262a4c1c0ea79c9043176d20,AGGACCGGATCAACTCCGGAGAGTCTCAGCTCCCGCAGCCCTAACA...,element,test,"candidate CRE nearby cardiac, neuro, cava and ...",hg38,NA,NA,NA,NA,NA,NA,NA,NA
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
80210,oligo_23474e71ce5c02846892d2e5f40bdfc4,AGGACCGGATCAACTCTTAATCAAATAACCCATTAATTCTATATAT...,NA,NA,NA,NA,NA,NA,NA,NA,NA,NA,NA,NA
80211,oligo_7cf2325b7ef95359164092b47379b3d5,AGGACCGGATCAACTCATCGGCCCTGGTGAAGCGTCCGTCCAGACG...,NA,NA,NA,NA,NA,NA,NA,NA,NA,NA,NA,NA
80212,oligo_db734997542b4f69c38d4d3c6e88603e,AGGACCGGATCAACTTAAATATTCAGCGATACATTCCTATTCTTTT...,NA,NA,NA,NA,NA,NA,NA,NA,NA,NA,NA,NA
80213,oligo_095855115099aa9e709dc0452fbc4edd,AGGACCGGATCAACTTGAAGCCCCTGATTCTGTTAGAATAAGGTTA...,NA,NA,NA,NA,NA,NA,NA,NA,NA,NA,NA,NA


In [8]:
pre_metadata_df

,header,sequence,id,category,class,source,ref_sequence,chrom,chrom_start,chrom_end,variant_class,variant_pos,SPDI,allele,info
0,cardiac_neuro_cava_random:SKI|ENSG00000157933....,AGGACCGGATCAACTAAGAATACAAGTAACTGATGAATGAAGGGGG...,oligo_c32acb98ad2a851ab46621b1c3af8b44,element,test,"candidate CRE nearby cardiac, neuro, cava and ...",hg38,NA,NA,NA,NA,NA,NA,NA,NA
1,cardiac_neuro_cava_random:SKI|ENSG00000157933....,AGGACCGGATCAACTTTGGGTATGCTGCCCCCCAGCTGGCGGGGCA...,oligo_8deb96fab75f4f41dbc05b2d163ad89b,element,test,"candidate CRE nearby cardiac, neuro, cava and ...",hg38,NA,NA,NA,NA,NA,NA,NA,NA
2,cardiac_neuro_cava_random:SKI|ENSG00000157933....,AGGACCGGATCAACTACGAGCAAGGGAATGAGAGAGAGTGGGTTAG...,oligo_d08942ae2ac12327ebaad04b395b7dc5,element,test,"candidate CRE nearby cardiac, neuro, cava and ...",hg38,NA,NA,NA,NA,NA,NA,NA,NA
3,cardiac_neuro_cava_random:SKI|ENSG00000157933....,AGGACCGGATCAACTCGTGGACACGCGTGATTGACCCTTTAACTGT...,oligo_d0ac046887b1f97d9c494e6a2bae69f7,element,test,"candidate CRE nearby cardiac, neuro, cava and ...",hg38,NA,NA,NA,NA,NA,NA,NA,NA
4,cardiac_neuro_cava_random:SKI|ENSG00000157933....,AGGACCGGATCAACTCCGGAGAGTCTCAGCTCCCGCAGCCCTAACA...,oligo_328edd51262a4c1c0ea79c9043176d20,element,test,"candidate CRE nearby cardiac, neuro, cava and ...",hg38,NA,NA,NA,NA,NA,NA,NA,NA
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
80210,MK:tile_2240|chr1-116244322+116244591|scramble...,AGGACCGGATCAACTCTTAATCAAATAACCCATTAATTCTATATAT...,oligo_23474e71ce5c02846892d2e5f40bdfc4,NA,NA,NA,NA,NA,NA,NA,NA,NA,NA,NA,NA
80211,MK:tile_6675|chr11-2374617+2374886|scramble_ne...,AGGACCGGATCAACTCATCGGCCCTGGTGAAGCGTCCGTCCAGACG...,oligo_7cf2325b7ef95359164092b47379b3d5,NA,NA,NA,NA,NA,NA,NA,NA,NA,NA,NA,NA
80212,MK:tile_18415|chr17-71181691+71181960|scramble...,AGGACCGGATCAACTTAAATATTCAGCGATACATTCCTATTCTTTT...,oligo_db734997542b4f69c38d4d3c6e88603e,NA,NA,NA,NA,NA,NA,NA,NA,NA,NA,NA,NA
80213,MK:tile_14356|chr15-67031618+67031887|scramble...,AGGACCGGATCAACTTGAAGCCCCTGATTCTGTTAGAATAAGGTTA...,oligo_095855115099aa9e709dc0452fbc4edd,NA,NA,NA,NA,NA,NA,NA,NA,NA,NA,NA,NA


In [19]:
# test category and variant_class
pre_metadata_df[pre_metadata_df['category'] == 'variant']

,header,sequence,id,category,class,source,ref_sequence,chrom,chrom_start,chrom_end,variant_class,variant_pos,SPDI,allele,info
8900,cardiac_neuro_cava_random:REF_SKI|ENSG00000157...,AGGACCGGATCAACTGTCCCAGCTCCCCACTGATGTGAAAGGTGGT...,oligo_17b5d1f079f92f165ae6b760da616314,variant,test,"candidate CRE nearby cardiac, neuro, cava and ...",hg38,NA,NA,NA,SNV,NA,NA,ref,NA
8901,cardiac_neuro_cava_random:REF_SKI|ENSG00000157...,AGGACCGGATCAACTCCTGATCTGCCCTGTCCGTGACGCTTCTGCT...,oligo_2e53f6e8c61ef7cd84a0c218519435a9,variant,test,"candidate CRE nearby cardiac, neuro, cava and ...",hg38,NA,NA,NA,SNV,NA,NA,ref,NA
8902,cardiac_neuro_cava_random:REF_SKI|ENSG00000157...,AGGACCGGATCAACTCCTCTGGGTGACCCGGAGAACACCAAGGCTG...,oligo_fa6b9ab66a2c9d8a83e0b807968d16de,variant,test,"candidate CRE nearby cardiac, neuro, cava and ...",hg38,NA,NA,NA,SNV,NA,NA,ref,NA
8903,cardiac_neuro_cava_random:REF_SKI|ENSG00000157...,AGGACCGGATCAACTCCATGCGGTGGCCACAGCCTCGGGTGAGTTC...,oligo_9be9e91086b215fc8a407ca5505bf01d,variant,test,"candidate CRE nearby cardiac, neuro, cava and ...",hg38,NA,NA,NA,SNV,NA,NA,ref,NA
8904,cardiac_neuro_cava_random:REF_SKI|ENSG00000157...,AGGACCGGATCAACTGGACTCCGGTGCCTTCGCATTCCCGAGCTGT...,oligo_512f507b4c554969462b4ce5da071afb,variant,test,"candidate CRE nearby cardiac, neuro, cava and ...",hg38,NA,NA,NA,SNV,NA,NA,ref,NA
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
73935,cardiac_neuro_cava_random:ALT_G6PD|ENSG0000016...,AGGACCGGATCAACTGGAGCTCTGCCTCACCCCACCTGGCCCCAAT...,oligo_6e74a25bd02b34708ccbf8093673ef62,variant,test,"candidate CRE nearby cardiac, neuro, cava and ...",hg38,NA,NA,NA,SNV,NA,NA,alt,NA
73936,cardiac_neuro_cava_random:ALT_G6PD|ENSG0000016...,AGGACCGGATCAACTATGTCTGAATTCACCTCCAAATAATGGGAAA...,oligo_87305b21de48ce90a3caf03cd0265009,variant,test,"candidate CRE nearby cardiac, neuro, cava and ...",hg38,NA,NA,NA,SNV,NA,NA,alt,NA
73937,cardiac_neuro_cava_random:ALT_G6PD|ENSG0000016...,AGGACCGGATCAACTCCTCTGCCCTCCCTGGCTTCTTCCCCTGTCC...,oligo_07249343e8d909bfd63abcc7ff28b50d,variant,test,"candidate CRE nearby cardiac, neuro, cava and ...",hg38,NA,NA,NA,SNV,NA,NA,alt,NA
73938,cardiac_neuro_cava_random:ALT_G6PD|ENSG0000016...,AGGACCGGATCAACTCCTCTGCCCTCCCTGGCTTCTTCCCCTGTCC...,oligo_e35878121d9479168994f561fc4a4028,variant,test,"candidate CRE nearby cardiac, neuro, cava and ...",hg38,NA,NA,NA,SNV,NA,NA,alt,NA


In [20]:
pre_metadata_df

,header,sequence,id,category,class,source,ref_sequence,chrom,chrom_start,chrom_end,variant_class,variant_pos,SPDI,allele,info
0,cardiac_neuro_cava_random:SKI|ENSG00000157933....,AGGACCGGATCAACTAAGAATACAAGTAACTGATGAATGAAGGGGG...,oligo_c32acb98ad2a851ab46621b1c3af8b44,element,test,"candidate CRE nearby cardiac, neuro, cava and ...",hg38,NA,NA,NA,NA,NA,NA,NA,NA
1,cardiac_neuro_cava_random:SKI|ENSG00000157933....,AGGACCGGATCAACTTTGGGTATGCTGCCCCCCAGCTGGCGGGGCA...,oligo_8deb96fab75f4f41dbc05b2d163ad89b,element,test,"candidate CRE nearby cardiac, neuro, cava and ...",hg38,NA,NA,NA,NA,NA,NA,NA,NA
2,cardiac_neuro_cava_random:SKI|ENSG00000157933....,AGGACCGGATCAACTACGAGCAAGGGAATGAGAGAGAGTGGGTTAG...,oligo_d08942ae2ac12327ebaad04b395b7dc5,element,test,"candidate CRE nearby cardiac, neuro, cava and ...",hg38,NA,NA,NA,NA,NA,NA,NA,NA
3,cardiac_neuro_cava_random:SKI|ENSG00000157933....,AGGACCGGATCAACTCGTGGACACGCGTGATTGACCCTTTAACTGT...,oligo_d0ac046887b1f97d9c494e6a2bae69f7,element,test,"candidate CRE nearby cardiac, neuro, cava and ...",hg38,NA,NA,NA,NA,NA,NA,NA,NA
4,cardiac_neuro_cava_random:SKI|ENSG00000157933....,AGGACCGGATCAACTCCGGAGAGTCTCAGCTCCCGCAGCCCTAACA...,oligo_328edd51262a4c1c0ea79c9043176d20,element,test,"candidate CRE nearby cardiac, neuro, cava and ...",hg38,NA,NA,NA,NA,NA,NA,NA,NA
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
80210,MK:tile_2240|chr1-116244322+116244591|scramble...,AGGACCGGATCAACTCTTAATCAAATAACCCATTAATTCTATATAT...,oligo_23474e71ce5c02846892d2e5f40bdfc4,NA,NA,NA,NA,NA,NA,NA,NA,NA,NA,NA,NA
80211,MK:tile_6675|chr11-2374617+2374886|scramble_ne...,AGGACCGGATCAACTCATCGGCCCTGGTGAAGCGTCCGTCCAGACG...,oligo_7cf2325b7ef95359164092b47379b3d5,NA,NA,NA,NA,NA,NA,NA,NA,NA,NA,NA,NA
80212,MK:tile_18415|chr17-71181691+71181960|scramble...,AGGACCGGATCAACTTAAATATTCAGCGATACATTCCTATTCTTTT...,oligo_db734997542b4f69c38d4d3c6e88603e,NA,NA,NA,NA,NA,NA,NA,NA,NA,NA,NA,NA
80213,MK:tile_14356|chr15-67031618+67031887|scramble...,AGGACCGGATCAACTTGAAGCCCCTGATTCTGTTAGAATAAGGTTA...,oligo_095855115099aa9e709dc0452fbc4edd,NA,NA,NA,NA,NA,NA,NA,NA,NA,NA,NA,NA


### Add knowledge on controls


In [13]:
pre_metadata_df[pre_metadata_df['ref_sequence'] != 'hg38']

# check which controls are element and variant 

# check which controlls are positive and negative (positive: for neuro positive, negative: for neuro negative)

# store headers to a tsv file:
header_path = 'controll_header.tsv'

controls = pre_metadata_df[pre_metadata_df['ref_sequence'] != 'hg38']
controls['header'].to_csv(header_path, sep='\t', index=False)

In [14]:
# checking manually: found many ALT_REF pairs
# - Weird: GC_Vista have ";" do not follow similar pattern (no ALT_ or REF_)
controls['label'] = controls['header'].str.split(':').str[0]
controls['label'].value_counts()
# get the different labels
positive_control_groups = ['C_positive_neuron_NP', 'C_positive_neuron_MK', 'C_positive_neuron_CD']

/tmp/ipykernel_2744/4031713056.py:3: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  controls['label'] = controls['header'].str.split(':').str[0]


label
MK                                   2397
C_positive_heart_AB                   909
GC_Selvarajan                         364
GC_Vista                              256
C_negative_heart_MK                   243
C_negative_neuron_MK                  222
C_negative_neuron_NP                  217
GC_Mendelian_variants                 209
GC_Kircher                            203
C_SLEA                                200
GC_Cort_Chengyu                       185
C_positive_neuron_NP                   99
C_positive_heart_CAD                   97
C_positive_heart_MK                    97
C_positive_neuron_MK                   96
C_positive_neuron_CD                   94
GC_GABA_Chengyu                        85
GC_DNase_positive_shuffeled            55
GC_Atrial_fib                          45
GC_DNase_positive                      41
GC_Glut_Chengyu                        40
GC_Mohlke                              34
GC_DNase_negative_blood_shuffeled      19
GC_Liang                    

## Start creating the Variant region lists:

In [5]:
# is there a header without "tile"? => No
design_df[(design_df['header'].str.contains("cardiac_neuro_cava_random") == True) & (design_df['header'].str.contains('tile') == False)]

,header,sequence,label


In [6]:
# filter for the rows with "cardiac_neuro_cava_random" in label column
cardiac_neuro_cava_random = design_df[design_df['label'] == 'cardiac_neuro_cava_random']

# do all of these rows have 2 pipes in the header?
cardiac_neuro_cava_random['header'].str.count(r'\|').value_counts()


# header
# 5     45082
# 2     27141
# 8       596
# 7       471
# 4       341
# 10      30


header
5     45082
2     27141
8       596
7       471
4       341
10      309
Name: count, dtype: int64

#### 2 pipe example
- 18340 REF
- 27141 all (if no REF_ or ALT_ then it is a region)
- example 
    - cardiac_neuro_cava_random:SKI|ENSG00000157933.11|EH38E2778476_fwd_tile1-1
    - cardiac_neuro_cava_random:REF_SKI|ENSG00000157933.11|EH38E2778534_fwd_tile1-1
- format: <label> : <sequence_type> _ <gene name> | <ensembl id> | <enhancer/encode id> _ <fwd/rev> _ <tile info>
- final columns: ['header', 'sequence', 'label', 'tile_info', 'strand', 'ensembl_id', 'enhancer_id', 'category', 'allele', 'gene_name']
- pattern = r'(?P<label>.*):(?P<allel>[A-Z]*_)?(?P<gene_name>.+)\|(?P<ensembl_id>.*?)\|(?P<enhancer_id>.*?)_(?P<fwd_rev>.*?)_(?P<tile_info>[a-z]+\d+[-]+\d+)'



In [17]:
# show me an example with threshold pipes in the header
num_pipes = 2
cardiac_neuro_cava_random[cardiac_neuro_cava_random['header'].str.count(r'\|') == num_pipes]['header'].tolist()[0]
# 'cardiac_neuro_cava_random:SKI|ENSG00000157933.11|EH38E2778476_fwd_tile1-1' # gene name | ensembl id | enhancer/encodeID_tile id

# does a row with 2 pipes in the header have ALT in the header?
cardiac_neuro_cava_random[cardiac_neuro_cava_random['header'].str.count(r'\|') == num_pipes]['header'].str.contains('ALT_').value_counts() # no alt there
cardiac_neuro_cava_random[cardiac_neuro_cava_random['header'].str.count(r'\|') == num_pipes]['header'].str.contains('REF_').value_counts() # no ref there # 18340 REF

# investigate the rows with 2 pipes in the header and ALT in the header
card_cava_2_pipes = cardiac_neuro_cava_random[cardiac_neuro_cava_random['header'].str.count(r'\|') == num_pipes]
print(card_cava_2_pipes.shape) # (27141, 3)
# card_cava_2_pipes['header'].str.contains('ALT')]['header'].tolist()[0]
card_cava_2_pipes[card_cava_2_pipes['header'].str.contains('REF_')]['header'].tolist()[10] # 'cardiac_neuro_cava_random:REF_SKI|ENSG00000157933.11|EH38E2778534_fwd_tile1-1'

(27141, 3)


'cardiac_neuro_cava_random:REF_SKI|ENSG00000157933.11|EH38E2778534_fwd_tile1-1'

##### split the dataframe 2 pipe 


In [18]:
# 1: pre_header column: header without the label
# 2: tile info column: tile info (split by "_" and take the last element) + remove this from the pre_header column
# 3: strand column: strand info (split by "_" and take again the last element) + remove this from the pre_header column
# 4: gene_name + ensembl_id + enhancer_id column: split the pre_header column by "|" and take the first 3 elements (expand = True)
# format: <label> : <gene name> | <ensembl id> | <enhancer/encode id> _ <fwd/rev> _ <tile info>

# short way using regex
pattern = r'(?P<label>.*):(?P<allel>[A-Z]*_)?(?P<gene_name>.+)\|(?P<ensembl_id>.*?)\|(?P<enhancer_id>.*?)_(?P<fwd_rev>.*?)_(?P<tile_info>[a-z]+\d+[-]+\d+)'
df_extracted = card_cava_2_pipes['header'].str.extract(pattern)
df_extracted
# concatenate df_extracted and card_cava_2_pipes
card_cava_2_pipes = pd.concat([card_cava_2_pipes, df_extracted], axis = 1)
card_cava_2_pipes

# long way: splitting manually
# # split the header column by ":" and take the second element
# card_cava_2_pipes['pre_header'] = card_cava_2_pipes['header'].str.split(':').str[1:].str.join(':')
# # tile info: split the pre_header column by "_" and take the last element
# card_cava_2_pipes['tile_info'] = card_cava_2_pipes['pre_header'].str.split('_').str[-1]
# # remove the tile info from the pre_header column
# card_cava_2_pipes['pre_header'] = card_cava_2_pipes['pre_header'].str.split('_').str[:-1].str.join('_')
# # strand info: split the pre_header column by "_" and take the last element
# card_cava_2_pipes['strand'] = card_cava_2_pipes['pre_header'].str.split('_').str[-1]
# # remove the strand info from the pre_header column
# card_cava_2_pipes['pre_header'] = card_cava_2_pipes['pre_header'].str.split('_').str[:-1].str.join('_')
# # gene_name + ensembl_id + enhancer_id: split the pre_header column by "|" and take the first 3 elements (expand = True)
# card_cava_2_pipes[['pre_gene_name', 'ensembl_id', 'enhancer_id']] = card_cava_2_pipes['pre_header'].str.split('|', expand = True)
# # remove the pre_header column
# card_cava_2_pipes.drop(columns = ['pre_header'], inplace = True)
# # category: element or variant
# card_cava_2_pipes['category'] = card_cava_2_pipes['pre_gene_name'].apply(lambda x: 'element' if '_' not in x else 'variant')
# # put alt or ref if "_"
# card_cava_2_pipes['allele'] = card_cava_2_pipes['pre_gene_name'].apply(lambda x: 'NA' if '_' not in x else x.split('_')[0].lower())
# # add gene_name column: split the pre_gene_name column by "_" and take the second element but only if "_" exists in the pre_gene_name column
# card_cava_2_pipes['gene_name'] = card_cava_2_pipes['pre_gene_name'].apply(lambda x: x.split('_')[1] if '_' in x else x)
# # drop the pre_gene_name column
# card_cava_2_pipes.drop(columns = ['pre_gene_name'], inplace = True)

# print(card_cava_2_pipes.columns)
# # ['header', 'sequence', 'label', 'tile_info', 'strand', 'ensembl_id', 'enhancer_id', 'category', 'allele', 'gene_name']
card_cava_2_pipes

,header,sequence,label,label,allel,gene_name,ensembl_id,enhancer_id,fwd_rev,tile_info
0,cardiac_neuro_cava_random:SKI|ENSG00000157933....,AGGACCGGATCAACTAAGAATACAAGTAACTGATGAATGAAGGGGG...,cardiac_neuro_cava_random,cardiac_neuro_cava_random,NaN,SKI,ENSG00000157933.11,EH38E2778476,fwd,tile1-1
1,cardiac_neuro_cava_random:SKI|ENSG00000157933....,AGGACCGGATCAACTTTGGGTATGCTGCCCCCCAGCTGGCGGGGCA...,cardiac_neuro_cava_random,cardiac_neuro_cava_random,NaN,SKI,ENSG00000157933.11,EH38E2778477,fwd,tile1-1
2,cardiac_neuro_cava_random:SKI|ENSG00000157933....,AGGACCGGATCAACTACGAGCAAGGGAATGAGAGAGAGTGGGTTAG...,cardiac_neuro_cava_random,cardiac_neuro_cava_random,NaN,SKI,ENSG00000157933.11,EH38E2778478,fwd,tile1-1
3,cardiac_neuro_cava_random:SKI|ENSG00000157933....,AGGACCGGATCAACTCGTGGACACGCGTGATTGACCCTTTAACTGT...,cardiac_neuro_cava_random,cardiac_neuro_cava_random,NaN,SKI,ENSG00000157933.11,EH38E2778480,fwd,tile1-1
4,cardiac_neuro_cava_random:SKI|ENSG00000157933....,AGGACCGGATCAACTCCGGAGAGTCTCAGCTCCCGCAGCCCTAACA...,cardiac_neuro_cava_random,cardiac_neuro_cava_random,NaN,SKI,ENSG00000157933.11,EH38E2778484,fwd,tile1-1
...,...,...,...,...,...,...,...,...,...,...
27477,cardiac_neuro_cava_random:REF_G6PD|ENSG0000016...,AGGACCGGATCAACTACCCCACTGCTGCACCAGATTGAGCTGGAGA...,cardiac_neuro_cava_random,cardiac_neuro_cava_random,REF_,G6PD,ENSG00000160211.20,EH38E3949715,rev,tile1-1
27478,cardiac_neuro_cava_random:REF_G6PD|ENSG0000016...,AGGACCGGATCAACTTTGCTGAGTAGTATCCGTTGTATGAATGCAC...,cardiac_neuro_cava_random,cardiac_neuro_cava_random,REF_,G6PD,ENSG00000160211.20,EH38E3949725,rev,tile1-1
27479,cardiac_neuro_cava_random:REF_G6PD|ENSG0000016...,AGGACCGGATCAACTGGAGCTCTGCCTCACCCCACCTGGCCCCAAT...,cardiac_neuro_cava_random,cardiac_neuro_cava_random,REF_,G6PD,ENSG00000160211.20,EH38E3949733,rev,tile1-1
27480,cardiac_neuro_cava_random:REF_G6PD|ENSG0000016...,AGGACCGGATCAACTATGTCTGAATTCACCTCCAAATAATGGGAAA...,cardiac_neuro_cava_random,cardiac_neuro_cava_random,REF_,G6PD,ENSG00000160211.20,EH38E2774396,rev,tile1-1


#### 4 pipe example (what group is this, mohan?)
- 242 REF / 99 (region)
- example: 
    - cardiac_neuro_cava_random:CSDE1|ENSG00000009307.17|EH38E1378377~NRAS|ENSG00000213281.5|EH38E1378377_rev_tile1-1
    - cardiac_neuro_cava_random:REF_CSDE1|ENSG00000009307.17|EH38E2832494~NRAS|ENSG00000213281.5|EH38E2832494_rev_tile1-1
- format: `<label> : [<sequence_type>_]*<gene name> | <ensembl id> | <enhancer/encode id> ~ <gene name> | <ensembl id> | <enhancer/encode id> _ <fwd/rev> _ <tile info>`
- assumption reference is also category variant
- final columns: ['header', 'sequence', 'label', 'additional_gene_association',
       'tile_info', 'strand', 'ensembl_id', 'enhancer_id', 'category',
       'allele', 'gene_name'] => additional gene association

In [13]:
num_pipes = 4
# Examples for the header string
# cardiac_neuro_cava_random:CSDE1|ENSG00000009307.17|EH38E1378377~NRAS|ENSG00000213281.5|EH38E1378377_rev_tile1-1
# cardiac_neuro_cava_random:REF_CSDE1|ENSG00000009307.17|EH38E2832494~NRAS|ENSG00000213281.5|EH38E2832494_rev_tile1-1
# format: `<label> : [<allele>_]*<gene name> | <ensembl id> | <enhancer/encode id> ~ <gene name> | <ensembl id> | <enhancer/encode id> _ <fwd/rev> _ <tile info>`
# Problem: allele is somethimes given (e.g. REF_ or "") but not everytime 
# solution regex: 
# pattern = r'(?P<label>.*):(?P<allel>[A-Z]*_)?(?P<gene_name>.+)\|(?P<ensembl_id>.*?)\|(?P<enhancer_id>.*?)_(?P<fwd_rev>.*?)_(?P<tile_info>.*?)_(?P<gene_name2>.*?)\|(?P<ensembl_id2>.*?)\|(?P<enhancer_id2>.*?)\|(?P<chrom2>.*?)-(?P<pos2>.*?)-(?P<ref2>.*?)-(?P<alt2>.*)'
pattern = r'(?P<label>.*):(?P<allel>[A-Z]*_)?(?P<gene_name>.+)\|(?P<ensembl_id>.*?)\|(?P<enhancer_id>.*?)~(?P<gene_name2>.+)\|(?P<ensembl_id2>.*?)\|(?P<enhancer_id2>.*?)_(?P<fwd_rev>.*?)_(?P<tile_info>.*)'
card_cava_4_pipes = cardiac_neuro_cava_random[cardiac_neuro_cava_random['header'].str.count(r'\|') == num_pipes]
# Extract values into new columns
df_extracted = card_cava_4_pipes['header'].str.extract(pattern)
df_extracted.head()

341


allel
REF_    242
Name: count, dtype: int64

In [14]:
# show me an example with threshold pipes in the header
num_pipes = 4
cardiac_neuro_cava_random[cardiac_neuro_cava_random['header'].str.count(r'\|') == num_pipes]['header'].tolist()[0]
# 'cardiac_neuro_cava_random:CSDE1|ENSG00000009307.17|EH38E1378377~NRAS|ENSG00000213281.5|EH38E1378377_rev_tile1-1'

card_cava_4_pipes = cardiac_neuro_cava_random[cardiac_neuro_cava_random['header'].str.count(r'\|') == num_pipes] # 341 rows
print(len(card_cava_4_pipes)) # 341
# do all these rows have ALT in the header?
card_cava_4_pipes['header'].str.contains('ALT_').value_counts() # no, no alt 
card_cava_4_pipes['header'].str.contains('REF_').value_counts() # True, 242 

# check the header of something containing REF_
card_cava_4_pipes[card_cava_4_pipes['header'].str.contains('REF_')]['header'].tolist()[0] # 'cardiac_neuro_cava_random:REF_CSDE1|ENSG00000009307.17|EH38E2832494~NRAS|ENSG00000213281.5|EH38E2832494_rev_tile1-1'

# check the header of something not containing REF_
card_cava_4_pipes[~card_cava_4_pipes['header'].str.contains('REF_')]['header'].tolist()[0] # 'cardiac_neuro_cava_random:CSDE1|ENSG00000009307.17|EH38E1378377~NRAS|ENSG00000213281.5|EH38E1378377_rev_tile1-1'

# check how many rows have ~ in the header
card_cava_4_pipes['header'].str.count('~').value_counts() # all have just one "~" 341

# check if all rows have 3 "_" in the header
card_cava_4_pipes['header'].str.count('_').value_counts() # all have 3 "_" 341

341


header
6    242
5     99
Name: count, dtype: int64

In [15]:
# format: `<label> : [<sequence_type>_]*<gene name> | <ensembl id> | <enhancer/encode id> ~ <gene name> | <ensembl id> | <enhancer/encode id> _ <fwd/rev> _ <tile info>`
# 1: pre_header column: header without the label
# 2: "additional_gene_association" column: split the pre_header column by "~" and take the second element
# 3: tile info column: tile info (split "additional_gene_association" by "_" and take the last element) + remove this from the column
# 4: strand info column: strand info (split "additional_gene_association" by "_" and take again the last element) + remove this from the column
# 5: remove the "additional_gene_association" column from the pre_header column (split by "~" and take the first element)
# 6: pre_gene_name + ensembl_id + enhancer_id: split the pre_header column by "|" and take the first 3 elements (expand = True)
# 7: remove the pre_header column
# 8: sequence_type: put "region" if pre_gene_name does not contain "_" otherwise split the pre_gene_name column by "_" and take the first element

# fast way using regex: 
pattern = r'(?P<label>.*):(?P<allel>[A-Z]*_)?(?P<gene_name>.+)\|(?P<ensembl_id>.*?)\|(?P<enhancer_id>.*?)~(?P<gene_name2>.+)\|(?P<ensembl_id2>.*?)\|(?P<enhancer_id2>.*?)_(?P<fwd_rev>.*?)_(?P<tile_info>.*)'
card_cava_4_pipes = cardiac_neuro_cava_random[cardiac_neuro_cava_random['header'].str.count(r'\|') == num_pipes]
df_extracted = card_cava_4_pipes['header'].str.extract(pattern)
print(df_extracted.head())

# # Concatenate the extracted columns with the original DataFrame
card_cava_4_pipes = pd.concat([card_cava_4_pipes, df_extracted], axis=1)
print(card_cava_4_pipes.head())

# long way: splitting manually
# # split the header column by ":" and take the second element
# card_cava_4_pipes['pre_header'] = card_cava_4_pipes['header'].str.split(':').str[1:].str.join(':')
# # additional_gene_association: split the pre_header column by "~" and take the second element
# card_cava_4_pipes['additional_gene_association'] = card_cava_4_pipes['pre_header'].str.split('~').str[1]
# # tile info: split the additional_gene_association column by "_" and take the last element
# card_cava_4_pipes['tile_info'] = card_cava_4_pipes['additional_gene_association'].str.split('_').str[-1]
# # remove the tile info from the additional_gene_association column
# card_cava_4_pipes['additional_gene_association'] = card_cava_4_pipes['additional_gene_association'].str.split('_').str[:-1].str.join('_')
# # strand info: split the additional_gene_association column by "_" and take the last element
# card_cava_4_pipes['strand'] = card_cava_4_pipes['additional_gene_association'].str.split('_').str[-1]
# # remove the strand info from the additional_gene_association column
# card_cava_4_pipes['additional_gene_association'] = card_cava_4_pipes['additional_gene_association'].str.split('_').str[:-1].str.join('_')
# # remove the additional_gene_association column from the pre_header column
# card_cava_4_pipes['pre_header'] = card_cava_4_pipes['pre_header'].str.split('~').str[0]
# # pre_gene_name + ensembl_id + enhancer_id: split the pre_header column by "|" and take the first 3 elements (expand = True)
# card_cava_4_pipes[['pre_gene_name', 'ensembl_id', 'enhancer_id']] = card_cava_4_pipes['pre_header'].str.split('|', expand = True)
# # remove the pre_header column
# card_cava_4_pipes.drop(columns = ['pre_header'], inplace = True)
# # category: element or variant
# card_cava_4_pipes['category'] = card_cava_4_pipes['pre_gene_name'].apply(lambda x: 'element' if '_' not in x else 'variant')
# # sequence_type: put "region" if pre_gene_name does not contain "_" otherwise split the pre_gene_name column by "_" and take the first element
# card_cava_4_pipes['allele'] = card_cava_4_pipes['pre_gene_name'].apply(lambda x: 'NA' if '_' not in x else x.split('_')[0].lower())

# # add gene_name column: split the pre_gene_name column by "_" and take the second element but only if "_" exists in the pre_gene_name column
# card_cava_4_pipes['gene_name'] = card_cava_4_pipes['pre_gene_name'].apply(lambda x: x.split('_')[1] if '_' in x else x)
# # drop the pre_gene_name column
# card_cava_4_pipes.drop(columns = ['pre_gene_name'], inplace = True)

# print(card_cava_4_pipes.columns)
# # ['header', 'sequence', 'label', 'additional_gene_association', 'tile_info', 'strand', 'ensembl_id', 'enhancer_id', 'category', 'allele', 'gene_name']
# card_cava_4_pipes


                         label allel gene_name          ensembl_id  \
708  cardiac_neuro_cava_random   NaN     CSDE1  ENSG00000009307.17   
709  cardiac_neuro_cava_random   NaN     CSDE1  ENSG00000009307.17   
710  cardiac_neuro_cava_random   NaN     CSDE1  ENSG00000009307.17   
711  cardiac_neuro_cava_random   NaN     CSDE1  ENSG00000009307.17   
712  cardiac_neuro_cava_random   NaN     CSDE1  ENSG00000009307.17   

      enhancer_id gene_name2        ensembl_id2  enhancer_id2 fwd_rev  \
708  EH38E1378377       NRAS  ENSG00000213281.5  EH38E1378377     rev   
709  EH38E2832502       NRAS  ENSG00000213281.5  EH38E2832502     rev   
710  EH38E2832508       NRAS  ENSG00000213281.5  EH38E2832508     rev   
711  EH38E2832513       NRAS  ENSG00000213281.5  EH38E2832513     rev   
712  EH38E1378387       NRAS  ENSG00000213281.5  EH38E1378387     rev   

    tile_info  
708   tile1-1  
709   tile1-1  
710   tile1-1  
711   tile1-1  
712   tile1-1  
                                            

enhancer id column; other genes with same enhancer
- mohan idea: 2D array or 3D array (gene column will have multiple genes)
- EH38E1378377~NRAS|ensemble id ... 
- fwd: (+ strand)
- rev: (- strand)
- gene name does not implicitly tell you the enhancer id (e.g. JUP|ENSG00000173801.17|EH38E3223379)

#### 5 pipe examples are all alt / variants (45082)
- 45082 alt
- example: 
    - cardiac_neuro_cava_random:ALT_SKI|ENSG00000157933.11|EH38E2778471_fwd_tile1-1_SKI|ENSG00000157933.11|EH38E2778471|1-2179591-T-C
    - cardiac_neuro_cava_random:ALT_ST3GAL3|ENSG00000126091.21|EH38E1342813_fwd_tile1-1_ST3GAL3|ENSG00000126091.21|EH38E1342813|1-43835787-G-A
    - cardiac_neuro_cava_random:ALT_IGLV3-25|ENSG00000211659.2|EH38E3470175_fwd_tile1-1_IGLV3-25|ENSG00000211659.2|EH38E3470175|22-22639049-T-C
- format: `<label> : <sequence_type (all ALT)> _ <gene name> | <ensembl id> | <enhancer/encode id> _ <fwd/rev> _ <tile info> _ <gene name> | <ensembl id> | <enhancer/encode id> | <variant info>`
- pattern = r'(?P<label>.*):ALT_(?P<gene_name>.*?)\|(?P<ensembl_id>.*?)\|(?P<enhancer_id>.*?)_(?P<fwd_rev>.*?)_(?P<tile_info>.*?)_(?P<gene_name2>.*?)\|(?P<ensembl_id2>.*?)\|(?P<enhancer_id2>.*?)\|(?P<chrom2>.*?)-(?P<pos2>.*?)-(?P<ref2>.*?)-(?P<alt2>.*)'

In [43]:
num_pipes = 5
# show me an example with 5 pipes in the header
cardiac_neuro_cava_random[cardiac_neuro_cava_random['header'].str.count(r'\|') == num_pipes]['header'].tolist()[2323]
# idx 0: cardiac_neuro_cava_random:ALT_SKI|ENSG00000157933.11|EH38E2778471_fwd_tile1-1_SKI|ENSG00000157933.11|EH38E2778471|1-2179591-T-C
# idx 10: cardiac_neuro_cava_random:ALT_SKI|ENSG00000157933.11|EH38E2778513_fwd_tile1-1_SKI|ENSG00000157933.11|EH38E2778513|1-2203222-G-A
# idx 24: cardiac_neuro_cava_random:ALT_SKI|ENSG00000157933.11|EH38E2778546_fwd_tile1-1_SKI|ENSG00000157933.11|EH38E2778546|1-2215527-T-C
# idx 2400: cardiac_neuro_cava_random:ALT_POMGNT1|ENSG00000085998.15|EH38E2809115_rev_tile1-1_POMGNT1|ENSG00000085998.15|EH38E2809115|1-46191391-T-C
# idx 2323: cardiac_neuro_cava_random:ALT_ST3GAL3|ENSG00000126091.21|EH38E1342813_fwd_tile1-1_ST3GAL3|ENSG00000126091.21|EH38E1342813|1-43835787-G-A


# investigate and find pattern
card_cava_5_pipes = cardiac_neuro_cava_random[cardiac_neuro_cava_random['header'].str.count(r'\|') == num_pipes] 
# does all of these rows have ALT_ after the first ":" in the header?
card_cava_5_pipes['header'].str.split(':').str[1].str.contains('ALT_').value_counts() # yes => all with 5 pipes have ALT_ in name (45082)
# card_cava_5_pipes['header'].str.split(':').str[1].str.contains('REF_').value_counts() #

# # do all of these rows end with the regex /-[A-Z]*-[A-Z]*/
# card_cava_5_pipes['header'].str.extract(r'(-[A-Z]*-[A-Z]*)$')[0].value_counts() # yes => all with 5 pipes have ALT_ in name

# # check if all rows have the same number of "-"
# card_cava_5_pipes['header'].str.count('-').value_counts() # 44237 have 4 and 845 have 6
# # check what the 6 "-" rows look like
# card_cava_5_pipes[card_cava_5_pipes['header'].str.count('-') == 6]['header'].tolist()[0] # cardiac_neuro_cava_random:ALT_IGLV3-25|ENSG00000211659.2|EH38E3470175_fwd_tile1-1_IGLV3-25|ENSG00000211659.2|EH38E3470175|22-22639049-T-C

# # investigate the different number of "-" cases
# print("header with 4 '-': ", card_cava_5_pipes[card_cava_5_pipes['header'].str.count(r'-') == 4]['header'].to_list()[4:8]) 
# print("header with 6 '-': ", card_cava_5_pipes[card_cava_5_pipes['header'].str.count(r'-') == 6]['header'].to_list()[10:14]) # some genes do have "-" in there name

# do all of them have 3 "_" between 2 and 3
# list(card_cava_5_pipes['header'].str.split('|'))


header
True    45082
Name: count, dtype: int64

In [33]:
# format: `<label> : <sequence_type (all ALT)> _ <gene name> | <ensembl id> | <enhancer/encode id> _ <fwd/rev> _ <tile info> _ <gene name> | <ensembl id> | <enhancer/encode id> | <variant info>`
# 1: pre_header column: header without the label
# 2: "additional_gene_association" column: split the pre_header column by "~" and take the second element
# 3: tile info column: tile info (split "additional_gene_association" by "_" and take the last element) + remove this from the column
# 4: strand info column: strand info (split "additional_gene_association" by "_" and take again the last element) + remove this from the column
# 5: remove the "additional_gene_association" column from the pre_header column (split by "~" and take the first element)
# 6: pre_gene_name + ensembl_id + enhancer_id: split the pre_header column by "|" and take the first 3 elements (expand = True)
# 7: remove the pre_header column
# 8: sequence_type: put "region" if pre_gene_name does not contain "_" otherwise split the pre_gene_name column by "_" and take the first element

# Define the regular expression pattern
pattern = r'(?P<label>.*):ALT_(?P<gene_name>.*?)\|(?P<ensembl_id>.*?)\|(?P<enhancer_id>.*?)_(?P<fwd_rev>.*?)_(?P<tile_info>.*?)_(?P<gene_name2>.*?)\|(?P<ensembl_id2>.*?)\|(?P<enhancer_id2>.*?)\|(?P<chrom2>.*?)-(?P<pos2>.*?)-(?P<ref2>.*?)-(?P<alt2>.*)'

# Extract values into new columns
df_extracted = card_cava_5_pipes['header'].str.extract(pattern)

# # Concatenate the extracted columns with the original DataFrame
card_cava_5_pipes = pd.concat([card_cava_5_pipes, df_extracted], axis=1)
print(card_cava_5_pipes.head())
# gene_name == gene_name2
## checking if same gene name or enhancer or ensembl id for all sequences => no, but for the majority
#! Might be interesting for the resulting table
# df_extracted["same_gene_names"] = df_extracted["gene_name"] == df_extracted["gene_name2"]
# print(len(df_extracted) - df_extracted["same_gene_names"].sum())
# df_extracted["same_enhancer"] = df_extracted["enhancer_id"] == df_extracted["enhancer_id2"] 
# print(len(df_extracted) - df_extracted["same_enhancer"].sum())
# df_extracted["same_ensemble"] = df_extracted["ensembl_id"] == df_extracted["ensembl_id2"]
# print(len(df_extracted) - df_extracted["same_ensemble"].sum())
# df_extracted[df_extracted["gene_name"] != df_extracted["gene_name2"]]

,label,gene_name,ensembl_id,enhancer_id,fwd_rev,tile_info,gene_name2,ensembl_id2,enhancer_id2,chrom2,pos2,ref2,alt2
27482,cardiac_neuro_cava_random,SKI,ENSG00000157933.11,EH38E2778471,fwd,tile1-1,SKI,ENSG00000157933.11,EH38E2778471,1,2179591,T,C
27483,cardiac_neuro_cava_random,SKI,ENSG00000157933.11,EH38E2778490,fwd,tile1-1,SKI,ENSG00000157933.11,EH38E2778490,1,2191444,G,A
27484,cardiac_neuro_cava_random,SKI,ENSG00000157933.11,EH38E2778492,fwd,tile1-1,SKI,ENSG00000157933.11,EH38E2778492,1,2192015,G,T
27485,cardiac_neuro_cava_random,SKI,ENSG00000157933.11,EH38E1311587,fwd,tile1-1,SKI,ENSG00000157933.11,EH38E1311587,1,2192366,T,G
27486,cardiac_neuro_cava_random,SKI,ENSG00000157933.11,EH38E2778494,fwd,tile1-1,SKI,ENSG00000157933.11,EH38E2778494,1,2193142,G,A


#### 7 pipe examples (471) 
- all alt 471
- example: 
    - cardiac_neuro_cava_random:ALT_CSDE1|ENSG00000009307.17|EH38E2832509~NRAS|ENSG00000213281.5|EH38E2832509_rev_tile1-1_NRAS|ENSG00000213281.5|EH38E2832509|1-114691158-A-G
    - cardiac_neuro_cava_random:ALT_DEAF1|ENSG00000177030.19|EH38E2937979~SLC25A22|ENSG00000177542.11|EH38E2937979_rev_tile1-1_SLC25A22|ENSG00000177542.11|EH38E2937979|11-745581-A-G
- "~" differenciates same region for different gene
- "_" differenciates same region for region and variant id
- variant is indicated by chr-pos-ref-alt

- format: `<label> : <sequence_type (all ALT)> _ <gene name> | <ensembl id> | <enhancer/encode id> ~ <gene name (additional gene association) | <ensembl id> | <enhancer/encode id> _ <fwd/rev> _ <tile info> _ <gene name> | <ensembl id> | <enhancer/encode id> | <variant info>`

- `pattern = r'(?P<label>.*):ALT_(?P<gene_name>.*?)\|(?P<ensembl_id>.*?)\|(?P<enhancer_id>.*?)~(?P<gene_name2>.*?)\|(?P<ensembl_id2>.*?)\|(?P<enhancer_id2>.*?)_(?P<fwd_rev>.*?)_(?P<tile_info>.*?)_(?P<gene_name3>.*?)\|(?P<ensembl_id3>.*?)\|(?P<enhancer_id3>.*?)\|(?P<chrom3>.*?)-(?P<pos3>.*?)-(?P<ref3>.*?)-(?P<alt3>.*)'` 

In [31]:
num_pipes = 7
# show me an example with num_pipes pipes in the header
cardiac_neuro_cava_random[cardiac_neuro_cava_random['header'].str.count(r'\|') == num_pipes]['header'].tolist()[23]
# 'cardiac_neuro_cava_random:ALT_CSDE1|ENSG00000009307.17|EH38E2832509~NRAS|ENSG00000213281.5|EH38E2832509_rev_tile1-1_NRAS|ENSG00000213281.5|EH38E2832509|1-114691158-A-G'
# 'cardiac_neuro_cava_random:ALT_DEAF1|ENSG00000177030.19|EH38E2937979~SLC25A22|ENSG00000177542.11|EH38E2937979_rev_tile1-1_SLC25A22|ENSG00000177542.11|EH38E2937979|11-745581-A-G'


# # investigate and find pattern
card_cava_7_pipes = cardiac_neuro_cava_random[cardiac_neuro_cava_random['header'].str.count(r'\|') == num_pipes] 
# # does all of these rows have ALT_ after the first ":" in the header?
# card_cava_7_pipes['header'].str.split(':').str[1].str.startswith('ALT_').value_counts() # 471
# # card_cava_7_pipes['header'].str.split(':').str[1].str.startswith('REF_').value_counts() # no ref

# does each row has one ~
print("Check if all have one ~:", sum(card_cava_7_pipes['header'].str.count('~') == 1)) # 471 all have one ~ between 2 and third "|"
print("Check if all have one ~ between second and third '|': ", sum(card_cava_7_pipes['header'].str.split(r'\|').str[2].str.count('~') == 1)) # 471 all have one ~ between 2 and third "|"

# how many contain "rev" and how many contain "fwd"
print("All headers with rev: ", sum(card_cava_7_pipes['header'].str.count('rev'))) # 392


# Define the regular expression pattern
pattern = r'(?P<label>.*):ALT_(?P<gene_name>.*?)\|(?P<ensembl_id>.*?)\|(?P<enhancer_id>.*?)~(?P<gene_name2>.*?)\|(?P<ensembl_id2>.*?)\|(?P<enhancer_id2>.*?)_(?P<fwd_rev>.*?)_(?P<tile_info>.*?)_(?P<gene_name3>.*?)\|(?P<ensembl_id3>.*?)\|(?P<enhancer_id3>.*?)\|(?P<chrom3>.*?)-(?P<pos3>.*?)-(?P<ref3>.*?)-(?P<alt3>.*)'

# Extract values into new columns
df_extracted = card_cava_7_pipes['header'].str.extract(pattern)
df_extracted.head()


Check if all have one ~: 471
Check if all have one ~ between second and third '|':  471
All headers with rev:  392


,label,gene_name,ensembl_id,enhancer_id,gene_name2,ensembl_id2,enhancer_id2,fwd_rev,tile_info,gene_name3,ensembl_id3,enhancer_id3,chrom3,pos3,ref3,alt3
30703,cardiac_neuro_cava_random,CSDE1,ENSG00000009307.17,EH38E2832509,NRAS,ENSG00000213281.5,EH38E2832509,rev,tile1-1,NRAS,ENSG00000213281.5,EH38E2832509,1,114691158,A,G
30705,cardiac_neuro_cava_random,CSDE1,ENSG00000009307.17,EH38E2832510,NRAS,ENSG00000213281.5,EH38E2832510,rev,tile1-1,NRAS,ENSG00000213281.5,EH38E2832510,1,114691688,G,A
30706,cardiac_neuro_cava_random,CSDE1,ENSG00000009307.17,EH38E2832510,NRAS,ENSG00000213281.5,EH38E2832510,rev,tile1-1,CSDE1,ENSG00000009307.17,EH38E2832510,1,114691802,G,C
30707,cardiac_neuro_cava_random,CSDE1,ENSG00000009307.17,EH38E2832511,NRAS,ENSG00000213281.5,EH38E2832511,rev,tile1-1,NRAS,ENSG00000213281.5,EH38E2832511,1,114692408,G,A
30708,cardiac_neuro_cava_random,CSDE1,ENSG00000009307.17,EH38E2832511,NRAS,ENSG00000213281.5,EH38E2832511,rev,tile1-1,NRAS,ENSG00000213281.5,EH38E2832511,1,114692413,C,T


#### 8 pipe example 596
- 596 alt
-'cardiac_neuro_cava_random:ALT_DRD4|ENSG00000069696.7|EH38E2937745_fwd_tile1-1_DEAF1|ENSG00000177030.19|EH38E2937745|11-596480-T-C~DRD4|ENSG00000069696.7|EH38E2937745|11-596480-T-C'
- 
- pattern = r'(?P<label>.*):ALT_(?P<gene_name>.*?)\|(?P<ensembl_id>.*?)\|(?P<enhancer_id>.*?)_(?P<fwd_rev>.*?)_(?P<tile_info>.*?)_(?P<gene_name2>.*?)\|(?P<ensembl_id2>.*?)\|(?P<enhancer_id2>.*?)\|(?P<chrom2>.*?)-(?P<pos2>.*?)-(?P<ref2>.*?)-(?P<alt2>.*)~(?P<gene_name3>.*?)\|(?P<ensembl_id3>.*?)\|(?P<enhancer_id3>.*?)\|(?P<chrom3>.*?)-(?P<pos3>.*?)-(?P<ref3>.*?)-(?P<alt3>.*)'


In [22]:
num_pipes = 8
# show me an example with num_pipes pipes in the header
cardiac_neuro_cava_random[cardiac_neuro_cava_random['header'].str.count(r'\|') == num_pipes]['header'].tolist()[200]
# 'cardiac_neuro_cava_random:ALT_DRD4|ENSG00000069696.7|EH38E2937745_fwd_tile1-1_DEAF1|ENSG00000177030.19|EH38E2937745|11-596480-T-C~DRD4|ENSG00000069696.7|EH38E2937745|11-596480-T-C'

# investigate and find pattern
card_cava_8_pipes = cardiac_neuro_cava_random[cardiac_neuro_cava_random['header'].str.count(r'\|') == num_pipes] 
# # does all of these rows have ALT_ after the first ":" in the header?
# card_cava_8_pipes['header'].str.split(':').str[1].str.startswith('ALT_').value_counts() # 596
# # card_cava_8_pipes['header'].str.split(':').str[1].str.startswith('REF_').value_counts() # no ref

# check if all have one ~
print("Check if all have one ~:", sum(card_cava_8_pipes['header'].str.count('~') == 1)) # 596 all have one ~ between 2 and third "|"
print("Check if all have one ~ between 5 and 6 '|': ", sum(card_cava_8_pipes['header'].str.split(r'\|').str[5].str.count('~') == 1)) # 596 all have one ~ 
# check if all have same number of "_"
print("Check if all have three _ between 2 and 3 '|': ", sum(card_cava_8_pipes['header'].str.split(r'\|').str[2].str.count('_') == 3))
print("Number of tile: ", sum(card_cava_8_pipes['header'].str.count('tile'))) # expected: 596 given: 596

# Define the regular expression pattern
pattern = r'(?P<label>.*):ALT_(?P<gene_name>.*?)\|(?P<ensembl_id>.*?)\|(?P<enhancer_id>.*?)_(?P<fwd_rev>.*?)_(?P<tile_info>.*?)_(?P<gene_name2>.*?)\|(?P<ensembl_id2>.*?)\|(?P<enhancer_id2>.*?)\|(?P<chrom2>.*?)-(?P<pos2>.*?)-(?P<ref2>.*?)-(?P<alt2>.*)~(?P<gene_name3>.*?)\|(?P<ensembl_id3>.*?)\|(?P<enhancer_id3>.*?)\|(?P<chrom3>.*?)-(?P<pos3>.*?)-(?P<ref3>.*?)-(?P<alt3>.*)'
# Extract values into new columns
df_extracted = card_cava_8_pipes['header'].str.extract(pattern)
df_extracted.head()

Check if all have one ~: 596
Check if all have one ~ between 5 and 6 '|':  596
Check if all have three _ between 2 and 3 '|':  596
Number of tile:  596


,label,gene_name,ensembl_id,enhancer_id,fwd_rev,tile_info,gene_name2,ensembl_id2,enhancer_id2,chrom2,pos2,ref2,alt2,gene_name3,ensembl_id3,enhancer_id3,chrom3,pos3,ref3,alt3
35409,cardiac_neuro_cava_random,DRD4,ENSG00000069696.7,EH38E2937745,fwd,tile1-1,DEAF1,ENSG00000177030.19,EH38E2937745,11,596480,T,C,DRD4,ENSG00000069696.7,EH38E2937745,11,596480,T,C
35410,cardiac_neuro_cava_random,DEAF1,ENSG00000177030.19,EH38E2937745,rev,tile1-1,DEAF1,ENSG00000177030.19,EH38E2937745,11,596480,T,C,DRD4,ENSG00000069696.7,EH38E2937745,11,596480,T,C
35411,cardiac_neuro_cava_random,DRD4,ENSG00000069696.7,EH38E2937745,fwd,tile1-1,DEAF1,ENSG00000177030.19,EH38E2937745,11,596499,G,C,DRD4,ENSG00000069696.7,EH38E2937745,11,596499,G,C
35412,cardiac_neuro_cava_random,DEAF1,ENSG00000177030.19,EH38E2937745,rev,tile1-1,DEAF1,ENSG00000177030.19,EH38E2937745,11,596499,G,C,DRD4,ENSG00000069696.7,EH38E2937745,11,596499,G,C
35415,cardiac_neuro_cava_random,DRD4,ENSG00000069696.7,EH38E2937745,fwd,tile1-1,DEAF1,ENSG00000177030.19,EH38E2937745,11,596672,G,T,DRD4,ENSG00000069696.7,EH38E2937745,11,596672,G,T


#### 10 pipes in example 309
- 309 alt
- example: 
  - 'cardiac_neuro_cava_random:ALT_CSDE1|ENSG00000009307.17|EH38E2832494~NRAS|ENSG00000213281.5|EH38E2832494_rev_tile1-1_CSDE1|ENSG00000009307.17|EH38E2832494|1-114668983-A-G~NRAS|ENSG00000213281.5|EH38E2832494|1-114668983-A-G'
  - 'cardiac_neuro_cava_random:ALT_CSDE1|ENSG00000009307.17|EH38E2832521~NRAS|ENSG00000213281.5|EH38E2832521_rev_tile1-1_CSDE1|ENSG00000009307.17|EH38E2832521|1-114716340-T-C~NRAS|ENSG00000213281.5|EH38E2832521|1-114716340-T-C'
- do all have "~" between "|" 2 and 3? yes
- do all have "_" between "|" 4 and 5? yes 
- do all have "~" between "|" 7 and 8? yes
- pattern = r'(?P<label>.*):ALT_(?P<gene_name>.*?)\|(?P<ensembl_id>.*?)\|(?P<enhancer_id>.*?)~(?P<gene_name2>.*?)\|(?P<ensembl_id2>.*?)\|(?P<enhancer_id2>.*?)_(?P<fwd_rev2>.*?)_(?P<tile_info2>.*?)_(?P<gene_name3>.*?)\|(?P<ensembl_id3>.*?)\|(?P<enhancer_id3>.*?)\|(?P<chrom3>.*?)-(?P<pos3>.*?)-(?P<ref3>.*?)-(?P<alt3>.*)~(?P<gene_name4>.*?)\|(?P<ensembl_id4>.*?)\|(?P<enhancer_id4>.*?)\|(?P<chrom4>.*?)-(?P<pos4>.*?)-(?P<ref4>.*?)-(?P<alt4>.*)'

In [30]:
num_pipes = 10
# show me an example with num_pipes pipes in the header
cardiac_neuro_cava_random[cardiac_neuro_cava_random['header'].str.count(r'\|') == num_pipes]['header'].tolist()[20]
# 'cardiac_neuro_cava_random:ALT_CSDE1|ENSG00000009307.17|EH38E2832494~NRAS|ENSG00000213281.5|EH38E2832494_rev_tile1-1_CSDE1|ENSG00000009307.17|EH38E2832494|1-114668983-A-G~NRAS|ENSG00000213281.5|EH38E2832494|1-114668983-A-G'
# 'cardiac_neuro_cava_random:ALT_CSDE1|ENSG00000009307.17|EH38E2832521~NRAS|ENSG00000213281.5|EH38E2832521_rev_tile1-1_CSDE1|ENSG00000009307.17|EH38E2832521|1-114716340-T-C~NRAS|ENSG00000213281.5|EH38E2832521|1-114716340-T-C'

# investigate and find pattern
card_cava_10_pipes = cardiac_neuro_cava_random[cardiac_neuro_cava_random['header'].str.count(r'\|') == num_pipes] 
# does all of these rows have ALT_ after the first ":" in the header?
card_cava_10_pipes['header'].str.split(':').str[1].str.startswith('ALT_').value_counts() # 309
# card_cava_10_pipes['header'].str.split(':').str[1].str.startswith('REF_').value_counts() # no ref

# - do all have "~" between "|" 2 and 3?
print("Check if all have one ~ between 2 and 3 '|': ", sum(card_cava_10_pipes['header'].str.split(r'\|').str[2].str.count('~') == 1)) # 309 all have one ~ 

# - do all have "_" between "|" 4 and 5?
print("Check if all have three _ between 4 and 5 '|': ", sum(card_cava_10_pipes['header'].str.split(r'\|').str[4].str.count('_') == 3))

# - do all have "~" between "|" 7 and 8? 
print("Check if all have one ~ between 7 and 8 '|': ", sum(card_cava_10_pipes['header'].str.split(r'\|').str[7].str.count('~') == 1)) # 309 all have one ~ 

# define regex
pattern = r'(?P<label>.*):ALT_(?P<gene_name>.*?)\|(?P<ensembl_id>.*?)\|(?P<enhancer_id>.*?)~(?P<gene_name2>.*?)\|(?P<ensembl_id2>.*?)\|(?P<enhancer_id2>.*?)_(?P<fwd_rev2>.*?)_(?P<tile_info2>.*?)_(?P<gene_name3>.*?)\|(?P<ensembl_id3>.*?)\|(?P<enhancer_id3>.*?)\|(?P<chrom3>.*?)-(?P<pos3>.*?)-(?P<ref3>.*?)-(?P<alt3>.*)~(?P<gene_name4>.*?)\|(?P<ensembl_id4>.*?)\|(?P<enhancer_id4>.*?)\|(?P<chrom4>.*?)-(?P<pos4>.*?)-(?P<ref4>.*?)-(?P<alt4>.*)'

# Extract values into new columns
df_extracted = card_cava_10_pipes['header'].str.extract(pattern)
df_extracted.head()

Check if all have one ~ between 2 and 3 '|':  309
Check if all have three _ between 4 and 5 '|':  309
Check if all have one ~ between 7 and 8 '|':  309


,label,gene_name,ensembl_id,enhancer_id,gene_name2,ensembl_id2,enhancer_id2,fwd_rev2,tile_info2,gene_name3,...,pos3,ref3,alt3,gene_name4,ensembl_id4,enhancer_id4,chrom4,pos4,ref4,alt4
30689,cardiac_neuro_cava_random,CSDE1,ENSG00000009307.17,EH38E2832494,NRAS,ENSG00000213281.5,EH38E2832494,rev,tile1-1,CSDE1,...,114668983,A,G,NRAS,ENSG00000213281.5,EH38E2832494,1,114668983,A,G
30690,cardiac_neuro_cava_random,CSDE1,ENSG00000009307.17,EH38E1378368,NRAS,ENSG00000213281.5,EH38E1378368,rev,tile1-1,CSDE1,...,114669283,T,C,NRAS,ENSG00000213281.5,EH38E1378368,1,114669283,T,C
30691,cardiac_neuro_cava_random,CSDE1,ENSG00000009307.17,EH38E2832496,NRAS,ENSG00000213281.5,EH38E2832496,rev,tile1-1,CSDE1,...,114670761,G,C,NRAS,ENSG00000213281.5,EH38E2832496,1,114670761,G,C
30692,cardiac_neuro_cava_random,CSDE1,ENSG00000009307.17,EH38E2832496,NRAS,ENSG00000213281.5,EH38E2832496,rev,tile1-1,CSDE1,...,114670764,T,C,NRAS,ENSG00000213281.5,EH38E2832496,1,114670764,T,C
30693,cardiac_neuro_cava_random,CSDE1,ENSG00000009307.17,EH38E2832496,NRAS,ENSG00000213281.5,EH38E2832496,rev,tile1-1,CSDE1,...,114670766,G,C,NRAS,ENSG00000213281.5,EH38E2832496,1,114670766,G,C


### Conclusion:

cardiac_neuro_cava_random:REF_SKI|ENSG00000157933.11|EH38E2778534_fwd_tile1-1
cardiac_neuro_cava_random:ALT_CSDE1|ENSG00000009307.17|EH38E2832494~NRAS|ENSG00000213281.5|EH38E2832494_rev_tile1-1_CSDE1|ENSG00000009307.17|EH38E2832494|1-114668983-A-G~NRAS|ENSG00000213281.5|EH38E2832494|1-114668983-A-G
cardiac_neuro_cava_random:ALT_DRD4|ENSG00000069696.7|EH38E2937745_fwd_tile1-1_DEAF1|ENSG00000177030.19|EH38E2937745|11-596480-T-C~DRD4|ENSG00000069696.7|EH38E2937745|11-596480-T-C
cardiac_neuro_cava_random:REF_CSDE1|ENSG00000009307.17|EH38E2832494~NRAS|ENSG00000213281.5|EH38E2832494_rev_tile1-1
cardiac_neuro_cava_random:CSDE1|ENSG00000009307.17|EH38E1378377~NRAS|ENSG00000213281.5|EH38E1378377_rev_tile1-1

<label>:[ALT_/REF_]*<gene_name>|<ensembl_id>|<enhancer/encodeID_tile id>[][~<gene_name>|<ensembl_id>|<enhancer/encodeID_tile id>]*_<fwd/rev>_<tile_info>

### Prepare a tsv of all labels and get the number of sequences

In [18]:
# write sequences with same label in one file
# get list of all labels in dataframe
labels = design_df['label'].unique().tolist()
group_list_output_dir = config['general']['group_lists_directory']

for label_of_interest in labels: 
    # get the rows of the dataframe with the label of interest
    label_of_interest_df = design_df[design_df['label'] == label_of_interest]
    # write it to directory as tsv file
    label_of_interest_df.to_csv(os.path.join(group_list_output_dir, label_of_interest + f'_{len(label_of_interest_df)}.tsv'), sep='\t', index=False)


In [19]:
# get number of all rows with a label starting with "C_"
C_labels = design_df[design_df['label'].str.startswith('C_')]
C_labels

,header,sequence,label
74811,C_positive_heart_CAD:REF_rs17114036,AGGACCGGATCAACTAGGAAGCAGGTCATAATTAGTGATAGTCATT...,C_positive_heart_CAD
74812,C_positive_heart_CAD:REF_rs72664324,AGGACCGGATCAACTTCCTCTGCTGAACCCACAGCAATGGCAGCCG...,C_positive_heart_CAD
74813,C_positive_heart_CAD:REF_rs12740374,AGGACCGGATCAACTTGACCCAAAAGTGCTTCATTTTTCGTGCCCG...,C_positive_heart_CAD
74814,C_positive_heart_CAD:REF_rs4450010,AGGACCGGATCAACTTGAGGTCCAAGGATGTGAGAGTGACCACAGT...,C_positive_heart_CAD
74815,C_positive_heart_CAD:REF_rs34091558,AGGACCGGATCAACTCTTCTCGGCCAATGAAGGGTCAACTCCATTG...,C_positive_heart_CAD
...,...,...,...
77723,C_SLEA:SLEA_hg18:chr9:82902419-82902586|6:V_Rx...,AGGACCGGATCAACTTAACTTCCAAGAGGCAGGGCCGTGACCCCGT...,C_SLEA
77724,C_SLEA:SLEA_hg18:chr9:82902419-82902586|7:V_AH...,AGGACCGGATCAACTTAACTTCCAAGAGGCAGCGGGGATCGCGTGC...,C_SLEA
77725,C_SLEA:SLEA_hg18:chr9:82902419-82902586|80:V_H...,AGGACCGGATCAACTTAACTTCCAAGAGGCAGCCAGGCAAGAAGTG...,C_SLEA
77726,C_SLEA:SLEA_hg18:chr9:82902419-82902586|8:V_HN...,AGGACCGGATCAACTTAACTTCCAAGAGGCAGCCAAGGTCCAGGTG...,C_SLEA


## Use vcf files to get the AF of the genes
- get the vcf files to the analysis (analyze_NGN2_feather.py)

## Use sequences of the deduplicated header and look for them in the duplicated header
- if you find a match take the header and make a list

In [20]:

# load duplicate and deduplicate fasta 
design_duplicates_fasta = "/home/kisa/coding/80K_MPRA/80K-Analysis/05_variant_region_list/resources/design.fa"
design_deduplicate_fasta = "/home/kisa/coding/80K_MPRA/80K-Analysis/05_variant_region_list/resources/design_no_duplicates_sequence_and_header.fa"
# load the fasta: 
design_dup_df = create_fasta_df_from_one_line_sequence_fasta(design_duplicates_fasta)
design_dedup_df = create_fasta_df_from_one_line_sequence_fasta(design_deduplicate_fasta)

In [72]:
design_dup_df["sequence"].astype(str)[15:285]

15     AGGACCGGATCAACTGGTTTGCTGTGGCCTGGCTCGATTGAGAATC...
16     AGGACCGGATCAACTCGGGATGGTGCCCGTGGCATCTTCTGCTCGG...
17     AGGACCGGATCAACTCTTGAACTCCTGACCTTGTGAGCTACCCACC...
18     AGGACCGGATCAACTGCAGTCTGCTTTTGGGCCTGTAGATTCGTTG...
19     AGGACCGGATCAACTCCACAGGCCAGGGCACTCCCAACAGCACTGC...
                             ...                        
280    AGGACCGGATCAACTGGGTCATCCAAGGTCCCAGGATCCAGCTCAT...
281    AGGACCGGATCAACTTGTGCGTGATCACCTGTGTAGCTCCTGGAGG...
282    AGGACCGGATCAACTTAAATTAAAATAAATAAACATTTAAAAATTA...
283    AGGACCGGATCAACTCATCAGAGTAGGTGAGACCACCTAGGGAAGA...
284    AGGACCGGATCAACTAGTACTTTGTGTTCTCATGTGACAATGGGCA...
Name: sequence, Length: 270, dtype: object

In [63]:
design_dup_df["sequence"] = design_dup_df["sequence"].astype(str)[15:285]
design_dedup_df["sequence"] = design_dedup_df["sequence"].astype(str)[15:285]

# # sort both files using their sequence
design_dup_df.sort_values(by=['sequence'], inplace=True)
design_dedup_df.sort_values(by=['sequence'], inplace=True)

In [64]:
design_dup_df

,header,sequence
227,cardiac_neuro_cava_random:NOS1AP|ENSG000001989...,AGGACCGGATCAACTAAAAATTTTTAAGGGAATTTTAAGTGTGAAA...
135,cardiac_neuro_cava_random:ST3GAL3|ENSG00000126...,AGGACCGGATCAACTAAAAGAGAGACAACTACTGCTTTTACTTCAG...
211,cardiac_neuro_cava_random:NOS1AP|ENSG000001989...,AGGACCGGATCAACTAAACAGCCATTACTACTTTAATAGAGCAGAG...
265,cardiac_neuro_cava_random:PBX1|ENSG00000185630...,AGGACCGGATCAACTAAAGAAGACAGATTATTCCCCAGTTGCCAAC...
124,cardiac_neuro_cava_random:ST3GAL3|ENSG00000126...,AGGACCGGATCAACTAAAGCTCTGGGGTGGGAAAAGGGCTCCCAAG...
...,...,...
80801,MK:tile_2240|chr1-116244322+116244591|scramble...,NaN
80802,MK:tile_6675|chr11-2374617+2374886|scramble_ne...,NaN
80803,MK:tile_18415|chr17-71181691+71181960|scramble...,NaN
80804,MK:tile_14356|chr15-67031618+67031887|scramble...,NaN


In [49]:
idx_dup = 0
idx_dedup = 0
duplicated_sequences_list = []
duplicate = False
while idx_dup < len(design_dup_df) and idx_dedup < len(design_dedup_df):
    if  design_dup_df.iloc[idx_dup]['sequence'] == design_dedup_df.iloc[idx_dedup]['sequence']:
        if (duplicate):
            # append to list
            duplicated_sequences_list[-1][2].append(design_dup_df.iloc[idx_dup]['header'])
        else:
            # create new list
            duplicated_sequences_list.append((design_dedup_df.iloc[idx_dedup]['header'], design_dedup_df.iloc[idx_dedup]['sequence'], [design_dup_df.iloc[idx_dup]['header']]))
            duplicate = True
        idx_dedup += 1
    elif design_dup_df.iloc[idx_dup]['sequence'] < design_dedup_df.iloc[idx_dedup]['sequence']:
        idx_dup += 1
        duplicate = False
    else:
        idx_dedup += 1
    
if len(design_dedup_df) != len(duplicated_sequences_list):
    for i in range(len(design_dedup_df)):
        if design_dedup_df[i] != duplicated_sequences_list[i][0]:
            print("Missing: " + str(design_dedup_df[i]))
            exit(1)

TypeError: '<' not supported between instances of 'str' and 'float'

In [43]:
duplicated_sequences_list

[('cardiac_neuro_cava_random:ALT_NCKAP1|ENSG00000061676.16|EH38E2057960_rev_tile1-1_NCKAP1|ENSG00000061676.16|EH38E2057960|2-183088247-A-T',
  'AGGACCGGATCAACTAAAAAAAAAACACAAAAAACAAAAAACAAAAAAAACCTCTTCCTTTGCTTCCATCCAAATACAGTCGTGCATCATTTAACAATGGTGATGCATTCTGAGAAATGTGTTGTTAGGCAGTTTCTTCGTGGTGTAAACATCACAGAGTGCACTCACACAAACCAAGAATGGTATGGCCTATTGCTCCAAGACTACAAACCTGTTCAGATGTTACTGTACTGACTTGTAGGCAACAGTAACACAATGGTATGTATTTGTGTATCTAAACATACACATTGCGTGAACCGA',
  ['cardiac_neuro_cava_random:ALT_NCKAP1|ENSG00000061676.16|EH38E2057960_rev_tile1-1_NCKAP1|ENSG00000061676.16|EH38E2057960|2-183088247-A-T']),
 ('cardiac_neuro_cava_random:REF_NCKAP1|ENSG00000061676.16|EH38E2057960_rev_tile1-1',
  'AGGACCGGATCAACTAAAAAAAAAACACAAAAAACAAAAAACAAAAAAAACCTCTTCCTTTGCTTCCATCCAAATACAGTCGTGCATCATTTAACAATGGTGATGCATTCTGAGAAATGTGTTGTTAGGCAGTTTCTTCGTGGTGTAAACATCACAGAGTGCACTCACACAAACCTAGAATGGTATGGCCTATTGCTCCAAGACTACAAACCTGTTCAGATGTTACTGTACTGACTTGTAGGCAACAGTAACACAATGGTATGTATTTGTGTATCTAAACATACACATTGCGTGAACCGA',
  ['cardiac_neuro

In [46]:
print(len(duplicated_sequences_list))

count_merge = 0
for tuple in duplicated_sequences_list:
    print(tuple[0])
    print(tuple[1])
    print(tuple[2])
    break
    if len(tuple[2]) >= 2:
        count_merge += 1
        
print("Mergings in the design: ", count_merge)

80215
cardiac_neuro_cava_random:ALT_NCKAP1|ENSG00000061676.16|EH38E2057960_rev_tile1-1_NCKAP1|ENSG00000061676.16|EH38E2057960|2-183088247-A-T
AGGACCGGATCAACTAAAAAAAAAACACAAAAAACAAAAAACAAAAAAAACCTCTTCCTTTGCTTCCATCCAAATACAGTCGTGCATCATTTAACAATGGTGATGCATTCTGAGAAATGTGTTGTTAGGCAGTTTCTTCGTGGTGTAAACATCACAGAGTGCACTCACACAAACCAAGAATGGTATGGCCTATTGCTCCAAGACTACAAACCTGTTCAGATGTTACTGTACTGACTTGTAGGCAACAGTAACACAATGGTATGTATTTGTGTATCTAAACATACACATTGCGTGAACCGA
['cardiac_neuro_cava_random:ALT_NCKAP1|ENSG00000061676.16|EH38E2057960_rev_tile1-1_NCKAP1|ENSG00000061676.16|EH38E2057960|2-183088247-A-T']
Mergings in the design:  0
